# Vaani SED V5 — complete pipeline

**What V5 changes** (everything else is your V3 network and `best_stage2.pt`):
1. **7 category frame tracks + a MIL clip head.** The single noise/no-noise output becomes one track per category. The new heads are copied from your trained head, so training starts from your model. A clip-level head pools those same tracks, so Silver and Bronze category labels teach category identity without trusting their timestamps.
2. **Corrected BEATs alignment.** Every window starts on the 0.16 s BEATs patch grid, and the cached rows are sliced so each row sits where the interpolation puts it (residual 7.5 ms, was up to −228 ms).
3. **Synthetic mixtures with exact labels.** Clean-speech stretches from Gold-train clips + pasted Gold-train noise events, so the model trains on the "clean speech + added noise" type that makes up ~38% of the test set. Built only from your training data; test audio is never used.
4. **Selection on natural + synthetic validation.** Checkpoints and the decoder are chosen with 0.62·Val-N + 0.38·Val-S and must hold on both district halves.

## How to run
1. Edit **V5 USER CONFIG** (next cell): `BASE_DIR`, `TEST_DIR`. Leave `ESC50_DIR = None` unless external data is allowed.
2. Recommended first: set `SMOKE = True`, Run All (a few minutes, writes only to `*_smoke` folders), check there are no errors, then set `SMOKE = False`.
3. Run All. Stages (each has a flag in the config cell):

| Stage | What it does | Time (RTX, estimate) | Output |
|---|---|---|---|
| E0 | best_stage2 through 3 inference paths on all Gold val | 20–40 min | `v5_work/e0_tracks.pkl`, verdict in log |
| E1 | 24,000 train mixtures + 1,000 Val-S, with BEATs | 1–2 h, resumable | `synth_v5/` |
| E2 | V5-A fine-tune, 8,000 updates, eval every 1,000 | ~2.5–3 h | `v5_work/v5a_ema.pt` / `_student.pt` |
| E3 | decoder selection + control comparison | 15–30 min | `v5_work/v5a_decoder.json`, verdict in log |
| Submit | test BEATs cache (explicit) + V5 zip | ~15 min | `submission/submission_v5a.zip` |
| E7 (off) | retrain on all Gold for the same update count | ~2.5 h | `submission/submission_v5a_final.zip` |

Everything is logged to `v5_work/v5_log.txt`; `v5_work/progress_live.txt` shows the current training step.

**Fallback:** `RUN_CONTROL_ZIP = True` re-creates the 0.9965 submission (cell-29 path) as `submission_0p9965_control.zip`. Submit V5 only if the E3 log line says it beats the control on both halves, and change one thing per leaderboard submission.


In [ ]:
# =============================================================================
# V5 USER CONFIG -- the only cell you should need to edit.
# Every stage below checks its RUN_* flag; a stage that is switched off is
# skipped cleanly, and stages that depend on it load its saved outputs from
# disk instead (so you can run the notebook in several sittings).
# =============================================================================
BASE_DIR = "C:/Users/231b258"                 # same folder as before (caches, best_stage2.pt)
TEST_DIR = f"{BASE_DIR}/test"
TEST_AUDIO_DIR = f"{TEST_DIR}/audio"
EXPECTED_TEST_CLIPS = 5517                    # hard check on the submission line count

# Optional external foregrounds for the synthetic mixtures. Leave None unless the
# competition rules allow external data. If set, point it at the unzipped ESC-50
# folder (the one containing audio/ and meta/esc50.csv).
ESC50_DIR = None

# ---- stages (see the run guide at the top) ----------------------------------
RUN_E0_RECONCILE      = False    # ~20-40 min: which inference path reproduces 0.9965?
RUN_CONTROL_ZIP       = False   # re-create the 0.9965 control submission (cell-29 path)
RUN_E0_TEST_PROBE     = False   # best_stage2 + corrected cached BEATs -> 1 LB probe zip
RUN_BUILD_SYNTH       = False    # synthetic train mixtures + Val-S (resumable)
RUN_TRAIN_V5          = True    # V5-A fine-tune from best_stage2.pt (~2.5-3 h)
RUN_DECODER_SELECTION = True    # CPU-only once val tracks are cached
RUN_TEST_SUBMISSION   = True    # builds the test BEATs cache (explicit) + V5 zip
RUN_FINAL_RETRAIN     = False   # E7: retrain on all Gold (train+val) at a fixed update count

SMOKE = False   # True = tiny end-to-end check (minutes). Never submit a SMOKE output.
RUN_TAG = "v5c_smel"
SEED = 42


## Part 1 — your V3 definitions (unchanged except: `BASE_DIR` now comes from the config cell)


In [ ]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


In [ ]:
# =============================================================================
# CELL: Setup  (LOCAL / WINDOWS -- native Windows, no WSL2, confirmed via
# `!uname -a` failing with "not recognized". No bash available, so every
# step below uses plain Python/subprocess instead of `!`-shell commands --
# these are the exact same operations, just OS-independent. Confirmed
# working stack: torch 2.14.0+cu132, CUDA 13.2, compute capability (12, 0)
# -- install with:
#   pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu130
# =============================================================================
import os, sys, random, json, re, csv, io, difflib, shutil, subprocess
from collections import Counter, defaultdict
from copy import deepcopy

# 387GB free on C:, so the home directory is fine -- no need to relocate
# anything. unilm/, beats_cache_full/, wav_cache_16k_flac/ will be created
# fresh here on first run.
# BASE_DIR is set in the V5 USER CONFIG cell
os.makedirs(BASE_DIR, exist_ok=True)

# ---- unilm/BEATs source: clone once, keep on re-runs (was `!rm -rf` +
# `!git clone` unconditionally every run -- that's wasteful even on Linux;
# now only clones if missing).
UNILM_DIR = f"{BASE_DIR}/unilm"
if not os.path.isdir(os.path.join(UNILM_DIR, "beats")):
    shutil.rmtree(UNILM_DIR, ignore_errors=True)
    if shutil.which("git") is None:
        raise FileNotFoundError(
            "git not found on PATH. Install Git for Windows "
            "(https://git-scm.com/download/win) and restart the kernel.")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/microsoft/unilm.git", UNILM_DIR], check=True)
else:
    print(f"unilm already present at {UNILM_DIR}, skipping clone")

# %pip (not !pip) targets THIS kernel's environment specifically -- on
# Windows especially, `!pip` can silently resolve to a different Python
# install than the one Jupyter is actually running.
%pip install -q scipy scikit-learn soundfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from sklearn.model_selection import GroupShuffleSplit
from scipy.signal import medfilt
import soundfile as sf
from datasets import load_from_disk, disable_caching

disable_caching()

BEATS_DIR = f"{UNILM_DIR}/beats"
if not os.path.isdir(BEATS_DIR):
    raise FileNotFoundError(f"BEATs source not found: {BEATS_DIR}")
sys.path.insert(0, BEATS_DIR)
from BEATs import BEATs, BEATsConfig

_dataset_root = f"{BASE_DIR}/archive/vaani_noise_dataset"
# Try the Kaggle-matching layout (.../vaani_noise_dataset/train) first;
# fall back to the root itself if this archive wasn't split into a `train`
# subfolder -- either way, DATASET_PATH ends up pointing at whichever
# directory actually has the load_from_disk arrow/json files in it.
DATASET_PATH = f"{_dataset_root}/train" if os.path.isdir(f"{_dataset_root}/train") else _dataset_root
BEATS_CHECKPOINT = f"{BASE_DIR}/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt"
if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f"DATASET_PATH not found: {DATASET_PATH} -- check the actual folder "
        f"name under {_dataset_root} (`dir \"{_dataset_root}\"`) and adjust.")
if not os.path.isfile(BEATS_CHECKPOINT):
    raise FileNotFoundError(f"BEATS_CHECKPOINT not found: {BEATS_CHECKPOINT} -- update this path")
print(f"DATASET_PATH resolved to: {DATASET_PATH}")
# BEATs cache: pure /kaggle/working, no packaged Kaggle Dataset involved.
# beats-cache-full-3 (a prior packaged upload) was confirmed 100% all-zero
# (every cached array read back as exact 0.0 -- almost certainly the old
# fp16-autocast NaN bug getting silently laundered into zeros by
# nan_to_num during a build that predates the fp32 fix). Rather than
# re-publish a Dataset, this rebuilds directly into the working directory
# each session and trains straight from that. KAGGLE_DATASET_CACHE_DIR is
# kept as a hook (None = unused) in case you want a packaged cache again
# later -- every downstream cell already reads BEATS_CACHE_DIRS as a list
# and degrades to a single directory when this is None, so nothing else
# needs to change if you do.
KAGGLE_DATASET_CACHE_DIR = None
BEATS_CACHE_BUILD_DIR = f"{BASE_DIR}/beats_cache_full"      # writable -- the only cache tier now
BEATS_CACHE_DIRS = ([KAGGLE_DATASET_CACHE_DIR] if KAGGLE_DATASET_CACHE_DIR else []) + [BEATS_CACHE_BUILD_DIR]
BEATS_CACHE_DIR = BEATS_CACHE_BUILD_DIR  # kept as an alias for any old single-dir usage

print("BEATs source:", BEATS_DIR)
print("Cache (read, priority order):", BEATS_CACHE_DIRS)
print("Cache (build/writable):", BEATS_CACHE_BUILD_DIR)
if KAGGLE_DATASET_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_CACHE_DIR):
    _n_packaged = len([f for f in os.listdir(KAGGLE_DATASET_CACHE_DIR) if f.endswith(".npy")])
    print(f"  packaged dataset cache: {_n_packaged} .npy files found at {KAGGLE_DATASET_CACHE_DIR}")
else:
    print("  no packaged dataset configured -- reading/writing BEATS_CACHE_BUILD_DIR only")

# Same pure-working-directory approach for the waveform cache (raw-audio
# decode+resample cache, separate from the BEATs feature cache above) --
# no zip/dataset publishing, just build once per session into
# WAV_CACHE_BUILD_DIR and read straight from it. KAGGLE_DATASET_WAV_CACHE_DIR
# is kept as an unused hook (None) for the same reason as its BEATs
# counterpart above.
KAGGLE_DATASET_WAV_CACHE_DIR = None
WAV_CACHE_BUILD_DIR = f"{BASE_DIR}/wav_cache_16k_flac"
WAV_CACHE_DIR = WAV_CACHE_BUILD_DIR  # alias, mirrors BEATS_CACHE_DIR
WAV_CACHE_DIRS = ([KAGGLE_DATASET_WAV_CACHE_DIR] if KAGGLE_DATASET_WAV_CACHE_DIR else []) + [WAV_CACHE_BUILD_DIR]
print("Waveform cache (read, priority order):", WAV_CACHE_DIRS)
if KAGGLE_DATASET_WAV_CACHE_DIR and os.path.isdir(KAGGLE_DATASET_WAV_CACHE_DIR):
    _n_wav_packaged = len([f for f in os.listdir(KAGGLE_DATASET_WAV_CACHE_DIR) if f.endswith(".flac")])
    print(f"  packaged waveform cache: {_n_wav_packaged} .flac files found at {KAGGLE_DATASET_WAV_CACHE_DIR}")
else:
    print("  no packaged waveform dataset configured -- reading/writing WAV_CACHE_BUILD_DIR only")

# BEATs fbank framing is 25 ms window / 10 ms hop, and patch_embedding has
# stride = input_patch_size = 16. So each BEATs patch advances 16 * 10 ms
# = 160 ms, with the first patch centered ~75 ms into the clip. These constants
# replace the previous `duration / T_full` approximation.
BEATS_FBANK_HOP_SEC    = 0.010
BEATS_PATCH_SIZE       = 16
BEATS_PATCH_HOP_SEC    = BEATS_FBANK_HOP_SEC * BEATS_PATCH_SIZE          # 0.16 s
BEATS_PATCH_OFFSET_SEC = (BEATS_PATCH_SIZE - 1) * BEATS_FBANK_HOP_SEC / 2  # 0.075 s
print(f"BEATs patch hop={BEATS_PATCH_HOP_SEC}s offset={BEATS_PATCH_OFFSET_SEC}s")


In [ ]:
# =============================================================================
# CELL: Tag canonicalization (unchanged)
# =============================================================================
_BRACKET_RE = re.compile(r"^[\[]?<(.+)>[\]]?$")
_SEED_SYNONYMS = {
    "crow cawaing": "crow cawing", "lip_smacking": "lip smacking",
    "cawaing": "cawing", "yawing": "yawning", "sniffling": "sniffing",
    "hiccups": "hiccup", "nose sneezing": "sneezing",
    "nose clearing": "throat clearing", "cleaning throat": "throat clearing",
    "throat cleaning": "throat clearing",
}
_INTERCHANGEABLE_SUFFIXES = {"noise", "sound", "voice", "playing", "tone", "song"}
_PLURAL_MAP = {"children": "child", "kids": "kid", "birds": "bird", "bugs": "bug",
               "insects": "insect", "vehicles": "vehicle", "songs": "song", "crickets": "cricket"}

def normalize_tag(raw_tag):
    m = _BRACKET_RE.match(raw_tag.strip())
    inner = m.group(1) if m else raw_tag.strip()
    inner = inner.replace("_", " ").lower().strip()
    inner = re.sub(r"\s+", " ", inner)
    return _SEED_SYNONYMS.get(inner, inner)

def _canonical_tokens(norm_tag):
    tokens = [_PLURAL_MAP.get(t, t) for t in norm_tag.split(" ")]
    core = [t for t in tokens if t not in _INTERCHANGEABLE_SUFFIXES]
    return tuple(core) if core else tuple(tokens)

def build_tag_clusters(tag_counts, similarity_threshold=0.82):
    normalized = {raw: normalize_tag(raw) for raw in tag_counts}
    token_key_to_raws = defaultdict(list)
    for raw, norm in normalized.items():
        token_key_to_raws[_canonical_tokens(norm)].append(raw)
    clusters = list(token_key_to_raws.values())
    keys = list(token_key_to_raws.keys())
    merged, used = [], [False] * len(clusters)
    for i in range(len(clusters)):
        if used[i]: continue
        group = list(clusters[i]); used[i] = True
        key_i = " ".join(keys[i])
        for j in range(i + 1, len(clusters)):
            if used[j]: continue
            if difflib.SequenceMatcher(None, key_i, " ".join(keys[j])).ratio() >= similarity_threshold:
                group.extend(clusters[j]); used[j] = True
        merged.append(group)
    raw_to_canonical, cluster_report = {}, []
    for group in merged:
        rep = max(group, key=lambda r: tag_counts.get(r, 0))
        canonical = normalize_tag(rep)
        total = sum(tag_counts.get(r, 0) for r in group)
        for r in group: raw_to_canonical[r] = canonical
        cluster_report.append({"canonical_tag": canonical, "total_count": total,
                               "n_raw_variants": len(group),
                               "raw_variants": sorted(group, key=lambda r: -tag_counts.get(r, 0))})
    cluster_report.sort(key=lambda r: -r["total_count"])
    return raw_to_canonical, cluster_report

def write_cluster_report_csv(cluster_report, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["canonical_tag", "total_count", "n_raw_variants", "raw_variants"])
        for row in cluster_report:
            w.writerow([row["canonical_tag"], row["total_count"], row["n_raw_variants"],
                        " | ".join(row["raw_variants"])])


In [ ]:
# =============================================================================
# CELL: Constants and helpers
# =============================================================================
COARSE_CATEGORIES = ["animal", "appliance_machine", "baby_child", "human_non_speech",
                     "phone_signal_alarm", "singing_music", "vehicle_traffic"]
COARSE_TO_IDX = {c: i for i, c in enumerate(COARSE_CATEGORIES)}

TIER_VERIFIED, TIER_UNVERIFIED, TIER_WEAK_ONLY, TIER_PSEUDO_BRONZE = 0, 1, 2, 3
_QUALITY_TO_TIER = {"verified_timestamps": TIER_VERIFIED,
                    "unverified_timestamps": TIER_UNVERIFIED,
                    "no_timestamps": TIER_WEAK_ONLY}

def split_indices_by_tier(dataset, indices=None):
    """Returns tier -> list of GLOBAL indices (positions into `dataset`)."""
    quality_col = dataset["annotationQuality"]
    idxs = range(len(dataset)) if indices is None else list(indices)
    gold, silver, bronze = [], [], []
    for i in idxs:
        q = quality_col[i]
        if q == "verified_timestamps": gold.append(i)
        elif q == "unverified_timestamps": silver.append(i)
        elif q == "no_timestamps": bronze.append(i)
    return {"gold": gold, "silver": silver, "bronze": bronze}

def _load_audio(audio_field):
    if audio_field.get("array") is not None:
        wav = np.asarray(audio_field["array"], dtype=np.float32)
        return wav, int(audio_field["sampling_rate"])
    if audio_field.get("bytes") is not None:
        wav, sr = sf.read(io.BytesIO(audio_field["bytes"]), dtype="float32")
    elif audio_field.get("path"):
        wav, sr = sf.read(audio_field["path"], dtype="float32")
    else:
        raise ValueError(f"Unknown audio field: {list(audio_field.keys())}")
    if wav.ndim > 1: wav = wav.mean(axis=1)
    return wav.astype(np.float32), int(sr)

def resolve_wav_cache_path(full_idx, cache_dir_or_dirs):
    """Same lookup pattern as resolve_cache_path (checks each dir in
    priority order, first match wins), but for the FLAC waveform cache --
    kept as a separate function rather than overloading resolve_cache_path
    so every existing BEATs-cache call site (.npy) is untouched."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.flac")
        if os.path.exists(p):
            return p
    return None

def _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir=None):
    """Loads a resampled mono waveform for full_idx. Uses the waveform
    cache when available (no audio decode, no row access at all); falls
    back to a live decode+resample otherwise. wav_cache_dir may be a
    single directory (old behavior) or a list -- read priority order,
    mirroring BEATS_CACHE_DIRS: e.g. a read-only packaged Kaggle Dataset
    checked first, the writable working-dir cache second. Self-healing
    writes always go to the LAST directory in the list, by convention the
    writable build dir (any earlier ones are read-only packaged data)."""
    dirs = _normalize_cache_dirs(wav_cache_dir)
    if dirs:
        p = resolve_wav_cache_path(full_idx, dirs)
        if p is not None:
            wav_np, _ = sf.read(p, dtype="float32")
            return torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    wav_np, sr = _load_audio(parent_hf[int(full_idx)]["audio"])
    wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
    if wav.ndim > 1: wav = wav.mean(dim=-1)
    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
    wav = wav.contiguous()
    if dirs:
        write_dir = dirs[-1]
        try:
            os.makedirs(write_dir, exist_ok=True)
            sf.write(os.path.join(write_dir, f"{int(full_idx):06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception:
            pass
    return wav

def build_label_table(hf_dataset):
    """Pulls ONLY the columns needed for labels via columnar access, once.
    This is what actually makes the waveform cache pay off: HF datasets
    decode the audio column as a side effect of ANY row access
    (hf_dataset[i]), even when you never touch that field -- so without
    this, every __getitem__ would still fully decode audio just to read
    NoiseCategory/NoiseSubCategoryTimeStamp/annotationQuality off the same
    row, then discard the decoded audio in favor of the cached waveform.
    Column access (hf_dataset["col"]) does not trigger that decode."""
    cols = hf_dataset.column_names
    return {
        "NoiseCategory": hf_dataset["NoiseCategory"],
        "NoiseSubCategoryTimeStamp": hf_dataset["NoiseSubCategoryTimeStamp"],
        "annotationQuality": hf_dataset["annotationQuality"],
        "id": hf_dataset["id"] if "id" in cols else [None] * len(hf_dataset),
    }

def _ref_events_for_full_idx(full_idx, label_table):
    evs = []
    for e in (label_table["NoiseSubCategoryTimeStamp"][full_idx] or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def _normalize_cache_dirs(cache_dir_or_dirs):
    if cache_dir_or_dirs is None: return []
    if isinstance(cache_dir_or_dirs, (list, tuple)): return list(cache_dir_or_dirs)
    return [cache_dir_or_dirs]

def resolve_cache_path(full_idx, cache_dir_or_dirs):
    """Checks each directory in priority order (e.g. a read-only packaged
    Kaggle Dataset first, the writable working-dir cache second) and
    returns the first existing path, or None if not found in any of them.
    Accepts either a single directory string (old behavior, unchanged) or
    a list of directories (new two-tier behavior)."""
    for d in _normalize_cache_dirs(cache_dir_or_dirs):
        p = os.path.join(d, f"{int(full_idx):06d}.npy")
        if os.path.exists(p):
            return p
    return None

def _validate_cache_exhaustive(all_required_indices, cache_dir, label=""):
    """Verify EVERY required index has a cache file in AT LEAST ONE of the
    given directories (str or list). Raises if any are missing everywhere.
    This is the only acceptable pre-training check -- sampling leaves room
    for thousands of silent mismatches."""
    all_required_indices = sorted({int(i) for i in all_required_indices})
    missing = [fi for fi in all_required_indices if resolve_cache_path(fi, cache_dir) is None]
    tag = f"[{label}] " if label else ""
    if missing:
        raise RuntimeError(
            f"{tag}BEATs cache incomplete: {len(missing)} of "
            f"{len(all_required_indices)} required indices missing from "
            f"{_normalize_cache_dirs(cache_dir)}. First 10: {missing[:10]}. "
            f"Rebuild the cache before training -- the notebook does not "
            f"fall back to live BEATs.")
    print(f"  {tag}[cache-validate] all {len(all_required_indices)} required "
          f"indices present across {_normalize_cache_dirs(cache_dir)}")
    return True


In [ ]:
# =============================================================================
# CELL: Mel frontend (unchanged)
# =============================================================================
class MelFrontend(nn.Module):
    def __init__(self, n_mels=128, sample_rate=16000, n_fft=2048,
                 hop_length=160, max_seconds=8.0):
        super().__init__()
        self.sample_rate = int(sample_rate); self.hop_length = int(hop_length)
        self.max_seconds = float(max_seconds); self.n_mels = int(n_mels)
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=n_fft, hop_length=self.hop_length,
            n_mels=self.n_mels, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
    def forward(self, waveform):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        max_len = int(round(self.max_seconds * self.sample_rate))
        if waveform.shape[1] > max_len: waveform = waveform[:, :max_len]
        elif waveform.shape[1] < max_len: waveform = F.pad(waveform, (0, max_len - waveform.shape[1]))
        mel = self.to_db(self.mel_spec(waveform))
        mean = mel.mean(dim=(1, 2), keepdim=True)
        std = mel.std(dim=(1, 2), keepdim=True).clamp_min(1e-5)
        return (mel - mean) / std

def apply_mel_specaugment(mel, time_masks=2, freq_masks=2, max_time_frac=0.08, max_freq_bins=12):
    if mel is None: return None
    out = mel.clone(); B, Freq, Time = out.shape
    max_t = max(1, int(Time * max_time_frac))
    for b in range(B):
        for _ in range(time_masks):
            w = random.randint(0, max_t)
            if w:
                s = random.randint(0, max(0, Time - w)); out[b, :, s:s+w] = 0
        for _ in range(freq_masks):
            w = random.randint(0, min(max_freq_bins, Freq))
            if w:
                s = random.randint(0, max(0, Freq - w)); out[b, s:s+w, :] = 0
    return out


In [ ]:
# =============================================================================
# CELL: DFD-LKA encoder
# =============================================================================
class DFDConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, n_kernels=4,
                 dilations=(1, 2, 3, 4)):
        super().__init__()
        self.kernel_size = kernel_size
        self.dilations = list(dilations)[:n_kernels]
        self.kernels = nn.ParameterList([
            nn.Parameter(torch.randn(out_ch, in_ch, kernel_size, kernel_size) * 0.02)
            for _ in self.dilations])
        self.biases = nn.ParameterList([nn.Parameter(torch.zeros(out_ch)) for _ in self.dilations])
        self.attn = nn.Sequential(nn.AdaptiveAvgPool2d((None, 1)),
                                  nn.Conv2d(in_ch, n_kernels, kernel_size=1))
    def forward(self, x):
        logits = self.attn(x).squeeze(-1)
        weights = F.softmax(logits, dim=1)
        H, W = x.shape[-2], x.shape[-1]
        out = 0.0
        for k, (kernel, bias, d) in enumerate(zip(self.kernels, self.biases, self.dilations)):
            pad = (self.kernel_size - 1) * d // 2
            y = F.conv2d(x, kernel, bias, padding=pad, dilation=d)
            if y.shape[-2] != H or y.shape[-1] != W: y = y[..., :H, :W]
            w = weights[:, k, :].unsqueeze(1).unsqueeze(-1)
            out = out + y * w
        return out

class LKAAttention(nn.Module):
    def __init__(self, channels, kernel_size=7, dilation=1):
        super().__init__()
        pad = (kernel_size // 2) * dilation
        self.dw = nn.Conv2d(channels, channels, kernel_size, padding=pad,
                            dilation=dilation, groups=channels)
        self.pw = nn.Conv2d(channels, channels, 1)
        self.gate = nn.Sigmoid()
    def forward(self, x):
        return x * self.gate(self.pw(self.dw(x))) + x

class DFDLKABlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_kernels=4, lka_kernel=7):
        super().__init__()
        self.dfd = DFDConv2d(in_ch, out_ch, n_kernels=n_kernels)
        self.bn = nn.BatchNorm2d(out_ch); self.act = nn.GELU()
        self.lka = LKAAttention(out_ch, kernel_size=lka_kernel)
    def forward(self, x):
        return self.lka(self.act(self.bn(self.dfd(x))))

class DFDLKAEncoder(nn.Module):
    def __init__(self, in_ch=1, channels=(32, 64, 128, 192),
                 freq_reduction=(2, 2, 2, 1), n_kernels=4, lka_kernel=7):
        super().__init__()
        self.blocks = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = in_ch
        for ch, fr in zip(channels, freq_reduction):
            self.blocks.append(DFDLKABlock(prev, ch, n_kernels, lka_kernel))
            self.pools.append(nn.MaxPool2d((fr, 1)) if fr > 1 else nn.Identity())
            prev = ch
        self.final_freq_pool = nn.AdaptiveAvgPool2d((1, None))
    def forward(self, x):
        for block, pool in zip(self.blocks, self.pools): x = pool(block(x))
        x = self.final_freq_pool(x).squeeze(2)
        return x.transpose(1, 2)

class TemporalRefinement(nn.Module):
    def __init__(self, dim, kernel_size=5):
        super().__init__()
        self.dw = nn.Conv1d(dim, dim, kernel_size, padding=kernel_size//2, groups=dim)
        self.norm = nn.LayerNorm(dim); self.act = nn.GELU()
    def forward(self, x):
        y = self.dw(x.transpose(1, 2)).transpose(1, 2)
        return self.act(self.norm(x + y))


In [ ]:
# =============================================================================
# CELL: BEATs temporal extractor + loader + cache builder
# =============================================================================
def beats_extract_temporal(beats_model, source, padding_mask=None,
                           fbank_mean=15.41663, fbank_std=6.55582):
    fbank = beats_model.preprocess(source, fbank_mean=fbank_mean, fbank_std=fbank_std)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(fbank, padding_mask)
    fbank = fbank.unsqueeze(1)
    feat2d = beats_model.patch_embedding(fbank)
    T_patches, F_patches = feat2d.shape[2], feat2d.shape[3]
    feat_seq = feat2d.reshape(feat2d.shape[0], feat2d.shape[1], -1).transpose(1, 2)
    feat_seq = beats_model.layer_norm(feat_seq)
    if padding_mask is not None:
        padding_mask = beats_model.forward_padding_mask(feat_seq, padding_mask)
    if beats_model.post_extract_proj is not None:
        feat_seq = beats_model.post_extract_proj(feat_seq)
    x = beats_model.dropout_input(feat_seq)
    x, _ = beats_model.encoder(x, padding_mask=padding_mask)
    x = x.reshape(x.shape[0], T_patches, F_patches, x.shape[-1])
    x_time = x.mean(dim=2)
    return x_time, T_patches, None

def load_beats_model(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    beats_cfg = BEATsConfig(checkpoint["cfg"])
    print("BEATs cfg: layers=", beats_cfg.encoder_layers,
          " dim=", beats_cfg.encoder_embed_dim,
          " patch=", getattr(beats_cfg, "input_patch_size", "n/a"))
    beats_model = BEATs(beats_cfg)
    beats_model.load_state_dict(checkpoint["model"], strict=True)
    beats_model.predictor = None
    beats_model.to(device).eval()
    for p in beats_model.parameters(): p.requires_grad_(False)
    print("BEATs loaded and frozen.")
    return beats_model, beats_cfg.encoder_embed_dim

@torch.no_grad()
def extract_beats_full(beats_model, waveform_1d, device):
    wav = waveform_1d.to(device).unsqueeze(0)
    # FIXED: was enabled=(device.type == "cuda"). Running a real, deep,
    # pretrained transformer under fp16 autocast produced all-NaN output
    # here (confirmed empirically -- fp16 gave NaN=True, fp32 gave a clean
    # min/max of -1.58/2.29 on the same clip). BEATs is frozen and this
    # only runs once per clip during caching, so there is no speed reason
    # to keep it in fp16 -- autocast is for things you backprop through.
    with autocast(device_type="cuda", enabled=False):
        feat, T_patches, _ = beats_extract_temporal(beats_model, wav)
    return feat[0].float().cpu().numpy(), T_patches

def build_beats_cache(hf_dataset, beats_model, device, cache_dir, sample_rate=16000,
                      max_clips=None, log_every=200, also_check_dirs=None):
    """Cache keyed by FULL-dataset index (i in range(len(hf_dataset))).
    Always WRITES to cache_dir (must be writable). also_check_dirs, if
    given, is a read-only-dirs-first check -- an index already present
    there (e.g. in a packaged Kaggle Dataset) is skipped entirely, so this
    never wastes time recomputing something you already have."""
    os.makedirs(cache_dir, exist_ok=True)
    n = len(hf_dataset) if max_clips is None else min(max_clips, len(hf_dataset))
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    manifest = {}
    n_skipped_from_other_dirs = 0
    for i in range(n):
        path = os.path.join(cache_dir, f"{i:06d}.npy")
        if resolve_cache_path(i, read_dirs) is not None:
            if not os.path.exists(path):
                n_skipped_from_other_dirs += 1
            continue
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, sr, sample_rate)
            feat, T_patches = extract_beats_full(beats_model, wav, device)

            # NEW: loud, unmissable warning if raw BEATs output is ever bad
            # again -- this is what should have existed the first time
            # instead of letting nan_to_num silently launder NaN into
            # deceptively clean-looking zeros with no trace left behind.
            if not np.isfinite(feat).all():
                print(f"  [cache] WARNING idx={i}: raw BEATs output has NaN/Inf before "
                      f"sanitizing ({np.isnan(feat).mean()*100:.1f}% NaN, "
                      f"{np.isinf(feat).mean()*100:.1f}% Inf)")

            # Kept as a backstop for genuinely corrupt individual clips only
            # -- with the fp32 fix above, this should now rarely if ever
            # actually trigger.
            feat32 = np.nan_to_num(feat.astype(np.float32), nan=0.0, posinf=6.0e4, neginf=-6.0e4)
            feat16 = np.clip(feat32, -6.0e4, 6.0e4).astype(np.float16)
            np.save(path, feat16)
            manifest[i] = {"T_patches": int(T_patches),
                           "duration": float(wav.numel() / sample_rate)}
        except Exception as e:
            print(f"  [cache] failed idx={i}: {e}")
        if (i + 1) % log_every == 0: print(f"  [cache] {i+1}/{n}")
    manifest_path = os.path.join(cache_dir, "manifest.json")
    existing = {}
    if os.path.exists(manifest_path):
        with open(manifest_path) as f: existing = json.load(f)
    existing.update({str(k): v for k, v in manifest.items()})
    with open(manifest_path, "w") as f: json.dump(existing, f)
    print(f"[cache] done. newly built here: {len(existing)}, "
          f"skipped ({n_skipped_from_other_dirs} already in another configured dir, "
          f"rest already in {cache_dir})")
    return manifest

# =============================================================================
# Waveform cache -- decode+resample once, reuse every epoch.
# Stored as FLAC (lossless, ~40-60% the size of raw float16 PCM) rather
# than .npy: the raw-array cache filled /kaggle/working's disk quota
# mid-build once already (16kHz float16 mono is 32KB/sec uncompressed --
# roughly 23GB across the full dataset), and FLAC keeps the same speed
# win (still skip decode+resample every epoch) at roughly half the disk
# footprint. Mirrors build_beats_cache's two-tier pattern exactly: skips
# indices already present in any read-only packaged dir (also_check_dirs),
# only writes to the given writable cache_dir. Unlike the BEATs cache, a
# missing entry here falls back gracefully to a live decode (see
# _load_waveform) rather than raising -- this cache is purely a speed
# optimization, not a correctness dependency.
# =============================================================================
def build_waveform_cache(hf_dataset, indices, cache_dir, sample_rate=16000,
                         log_every=2000, also_check_dirs=None):
    os.makedirs(cache_dir, exist_ok=True)
    indices = sorted({int(i) for i in indices})
    read_dirs = _normalize_cache_dirs(also_check_dirs) + [cache_dir]
    todo = [i for i in indices if resolve_wav_cache_path(i, read_dirs) is None]
    n_skipped = len(indices) - len(todo)
    print(f"  [wav-cache] {len(todo)} of {len(indices)} indices need caching "
          f"({n_skipped} already present in {read_dirs})")
    n_failed = 0
    for n, i in enumerate(todo):
        try:
            wav_np, sr = _load_audio(hf_dataset[i]["audio"])
            wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
            if wav.ndim > 1: wav = wav.mean(dim=-1)
            if sr != sample_rate:
                wav = torchaudio.functional.resample(wav, int(sr), sample_rate)
            sf.write(os.path.join(cache_dir, f"{i:06d}.flac"),
                     wav.numpy(), sample_rate, format="FLAC", subtype="PCM_16")
        except Exception as e:
            n_failed += 1
            print(f"  [wav-cache] failed idx={i}: {e}")
        if (n + 1) % log_every == 0:
            print(f"    [wav-cache] {n + 1}/{len(todo)} cached ({n_failed} failed so far)")
    print(f"  [wav-cache] done: {cache_dir} ({n_failed} failed, {n_skipped} skipped)")


In [ ]:
# =============================================================================
# CELL: VaaniNoiseDataset + collate_fn
#
# - Global-index invariant: cache + HF row indexed by same integer.
# - Short-clip BEATs padded to canonical 8s length; per-sample valid ratio
#   propagates so fusion can gate padded frames.
# - Patch-time slicing uses BEATs' fixed hop/offset, not duration/T_full.
# - STRICT lookup: missing cache raises, never silent zero-fill.
# =============================================================================
class VaaniNoiseDataset(Dataset):
    def __init__(self, parent_hf, full_indices, raw_to_canonical, canonical_tags,
                 fine_to_coarse_idx, beats_cache_dir=None, wav_cache_dir=None,
                 label_table=None, pseudo_overrides=None, crop_seconds=8.0, sample_rate=16000,
                 training=True, background_crop_prob=0.3, event_jitter_seconds=1.5):
        self.parent = parent_hf
        self.full_indices = np.asarray(full_indices, dtype=np.int64)
        assert self.full_indices.ndim == 1 and len(self.full_indices) > 0
        assert int(self.full_indices.max()) < len(parent_hf), \
            f"full_indices out of range: max={int(self.full_indices.max())} parent={len(parent_hf)}"
        self.raw_to_canonical = raw_to_canonical
        self.canonical_tags = canonical_tags
        self.tag_to_idx = {t: i for i, t in enumerate(canonical_tags)}
        self.fine_to_coarse_idx = fine_to_coarse_idx
        self.pseudo_overrides = pseudo_overrides or {}
        self.crop_seconds = float(crop_seconds); self.sample_rate = int(sample_rate)
        self.crop_samples = int(round(self.crop_seconds * self.sample_rate)) if self.crop_seconds > 0 else 0
        self.training = bool(training)
        self.background_crop_prob = float(background_crop_prob)
        self.event_jitter_seconds = float(event_jitter_seconds)
        self.beats_cache_dir = beats_cache_dir
        # NEW: wav_cache_dir avoids re-decoding audio every epoch;
        # label_table avoids the row-access that would otherwise decode
        # audio anyway just to read labels off the same row. Both optional
        # and independently gracefully-degrading -- omit either and this
        # dataset behaves exactly as before, just slower.
        self.wav_cache_dir = wav_cache_dir
        self.label_table = label_table
        self.canonical_T = (max(1, int(round(self.crop_seconds / BEATS_PATCH_HOP_SEC)))
                            if self.crop_seconds > 0 else 1)
        self._debug_prints_left = 3

    def __len__(self):
        return len(self.full_indices)

    def _full_idx(self, local_idx):
        return int(self.full_indices[local_idx])

    def _cache_path(self, full_idx):
        """Display/debug only -- returns the resolved path if found in any
        configured cache dir, else the join against the first dir (so a
        MISSING-file print still shows a sensible path, not None)."""
        found = resolve_cache_path(full_idx, self.beats_cache_dir)
        if found is not None:
            return found
        first_dir = _normalize_cache_dirs(self.beats_cache_dir)
        return os.path.join(first_dir[0], f"{int(full_idx):06d}.npy") if first_dir else "n/a"

    def _fine_idx(self, raw_tag):
        canon = self.raw_to_canonical.get(raw_tag, normalize_tag(raw_tag))
        return self.tag_to_idx.get(canon)

    def _resample_mono(self, wav_np, sr):
        wav = torch.from_numpy(np.asarray(wav_np, dtype=np.float32))
        if wav.ndim > 1: wav = wav.mean(dim=-1)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, int(sr), self.sample_rate)
        return wav.contiguous()

    def _load_beats_full(self, full_idx):
        if self.beats_cache_dir is None:
            raise RuntimeError("beats_cache_dir is required for this dataset.")
        path = resolve_cache_path(full_idx, self.beats_cache_dir)
        if path is None:
            raise RuntimeError(
                f"Missing BEATs cache entry for full_idx={full_idx} in "
                f"{_normalize_cache_dirs(self.beats_cache_dir)}. "
                f"Run build_beats_cache before training.")
        return np.load(path).astype(np.float32)

    def _slice_beats_crop(self, beats_full, start_s, end_s):
        """Patch-time slicing using BEATs' fixed hop/offset."""
        T_full = beats_full.shape[0]
        p0 = int(np.floor((start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
        p1 = int(np.ceil((end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
        p0 = max(0, min(p0, T_full - 1))
        p1 = max(p0 + 1, min(p1, T_full))
        return beats_full[p0:p1]

    def _pad_or_truncate_to_canonical(self, bw):
        if bw is None: return None
        if bw.shape[0] < self.canonical_T:
            pad = np.zeros((self.canonical_T - bw.shape[0], bw.shape[1]), dtype=bw.dtype)
            return np.concatenate([bw, pad], axis=0)
        if bw.shape[0] > self.canonical_T:
            return bw[:self.canonical_T]
        return bw

    def _select_crop(self, wav, event_spans, beats_full):
        n = wav.numel(); full_duration = n / self.sample_rate
        if self.crop_seconds <= 0:
            return wav, event_spans, full_duration, beats_full, 1.0
        target = self.crop_samples

        # -------- Short clip: pad waveform AND pad BEATs to canonical length --
        if n <= target:
            valid_seconds = n / self.sample_rate
            wav_p = F.pad(wav, (0, target - n))
            if beats_full is not None:
                bw = self._pad_or_truncate_to_canonical(beats_full)
                valid_ratio = min(1.0, valid_seconds / self.crop_seconds)
            else:
                bw = None; valid_ratio = 0.0
            return wav_p, event_spans, valid_seconds, bw, valid_ratio

        # -------- Normal crop inside a longer recording ----------------------
        take_bg = (not event_spans) or (random.random() < self.background_crop_prob)
        if take_bg:
            start = random.randint(0, n - target)
        else:
            s0, e0 = random.choice(event_spans)
            center = (s0 + e0) * 0.5 + random.uniform(-self.event_jitter_seconds, self.event_jitter_seconds)
            start = int(round(center * self.sample_rate - target / 2))
            start = max(0, min(start, n - target))
        end = start + target
        crop = wav[start:end]
        cs = start / self.sample_rate; ce = end / self.sample_rate
        cropped = []
        for s, e in event_spans:
            ns = max(0.0, float(s) - cs); ne = min(self.crop_seconds, float(e) - cs)
            if ne > ns: cropped.append((ns, ne))
        if beats_full is not None:
            bw = self._slice_beats_crop(beats_full, cs, ce)
            bw = self._pad_or_truncate_to_canonical(bw)
            valid_ratio = 1.0
        else:
            bw = None; valid_ratio = 0.0
        return crop, cropped, self.crop_seconds, bw, valid_ratio

    def __getitem__(self, idx, _retries=0):
        full_idx = self._full_idx(idx)
        if self._debug_prints_left > 0:
            print(f"    [dataset] local={idx} full_idx={full_idx} "
                  f"cache={self._cache_path(full_idx) if self.beats_cache_dir else 'n/a'}")
            self._debug_prints_left -= 1
        try:
            # CHANGED: was `self.parent[full_idx]` + `_load_audio` +
            # `_resample_mono` every single call. Uses the waveform cache
            # when available (no decode at all); falls back to a live
            # decode only for indices the cache doesn't have yet.
            wav = _load_waveform(full_idx, self.parent, self.sample_rate, self.wav_cache_dir)
        except Exception as exc:
            if _retries >= 5: raise RuntimeError(f"Failed to load audio full_idx={full_idx}") from exc
            return self.__getitem__(random.randrange(len(self)), _retries + 1)

        # CHANGED: was `sample = self.parent[full_idx]` then read fields off
        # it -- that row access decodes the audio column as a side effect
        # regardless of whether it's used, which would have made the
        # waveform cache above pointless. label_table (columnar, built
        # once) avoids that entirely; parent_hf row access is now only a
        # fallback for the rare case label_table wasn't provided.
        if self.label_table is not None:
            cats = self.label_table["NoiseCategory"][full_idx] or []
            raw_events = self.label_table["NoiseSubCategoryTimeStamp"][full_idx] or []
            quality = self.label_table["annotationQuality"][full_idx]
        else:
            sample = self.parent[full_idx]
            cats = sample.get("NoiseCategory") or []
            raw_events = sample.get("NoiseSubCategoryTimeStamp") or []
            quality = sample.get("annotationQuality")

        coarse_target = torch.zeros(len(COARSE_CATEGORIES), dtype=torch.float32)
        for cat in cats:
            if cat in COARSE_TO_IDX: coarse_target[COARSE_TO_IDX[cat]] = 1.0
        tier = _QUALITY_TO_TIER.get(quality, TIER_WEAK_ONLY)
        override = self.pseudo_overrides.get(full_idx)
        fine_clip_target = torch.zeros(len(self.canonical_tags), dtype=torch.float32)
        event_spans = []
        if override is not None:
            tier = TIER_PSEUDO_BRONZE
            for ev in override.get("events", []):
                try: s = float(ev["start"]); e = float(ev["end"])
                except (TypeError, ValueError, KeyError): continue
                if e > s: event_spans.append((s, e))
        else:
            for e in raw_events:
                try: s = float(e["start"]); en = float(e["end"])
                except (TypeError, ValueError, KeyError): continue
                if en <= s: continue
                event_spans.append((s, en))
                fi = self._fine_idx(e.get("tag", ""))
                if fi is not None: fine_clip_target[fi] = 1.0

        beats_full = self._load_beats_full(full_idx)   # STRICT

        if self.training and self.crop_seconds > 0:
            waveform, event_spans, valid_seconds, beats_crop, beats_valid_ratio = \
                self._select_crop(wav, event_spans, beats_full)
        else:
            waveform = wav
            valid_seconds = wav.numel() / self.sample_rate
            beats_crop = self._pad_or_truncate_to_canonical(beats_full)
            beats_valid_ratio = 1.0

        event_clip_target = torch.tensor(
            1.0 if (len(event_spans) > 0 or coarse_target.sum() > 0 or fine_clip_target.sum() > 0) else 0.0,
            dtype=torch.float32)

        return {
            "waveform": waveform.float().unsqueeze(0),
            "beats_crop": torch.from_numpy(np.asarray(beats_crop, dtype=np.float32)),
            "beats_valid_ratio": float(beats_valid_ratio),
            "duration": float(valid_seconds),
            "coarse_target": coarse_target,
            "fine_clip_target": fine_clip_target,
            "event_clip_target": event_clip_target,
            "event_spans": event_spans,
            "tier": int(tier),
            "src_idx": int(idx),
            "full_idx": int(full_idx),
        }

    def verify_cache_alignment(self, n_samples=5):
        if self.beats_cache_dir is None:
            print("  [cache-verify] no cache dir"); return
        k = min(n_samples, len(self))
        check = random.sample(range(len(self)), k) if len(self) > k else list(range(len(self)))
        print(f"  [cache-verify] checking {k}/{len(self)} samples "
              f"(expected canonical_T={self.canonical_T})")
        for li in check:
            fi = self._full_idx(li)
            cp = resolve_cache_path(fi, self.beats_cache_dir)
            if cp is None:
                print(f"    MISSING: local={li} full_idx={fi}"); continue
            feat = np.load(cp, mmap_mode="r")
            wav_np, sr = _load_audio(self.parent[fi]["audio"])
            dur = wav_np.shape[0] / sr
            expected_T = max(1, int(round(dur / BEATS_PATCH_HOP_SEC)))
            ratio = feat.shape[0] / expected_T
            flag = "OK" if 0.85 < ratio < 1.15 else "SUSPECT"
            print(f"    {flag}: local={li} full_idx={fi} dur={dur:.2f}s "
                  f"T_cached={feat.shape[0]} expected~{expected_T} ratio={ratio:.2f}")


def collate_fn(batch):
    waveforms = torch.cat([b["waveform"] for b in batch], dim=0)
    beats_list = [b["beats_crop"] for b in batch]
    T_max = max(f.shape[0] for f in beats_list)
    D = beats_list[0].shape[1]
    beats_padded = torch.zeros(len(batch), T_max, D, dtype=torch.float32)
    beats_mask = torch.zeros(len(batch), T_max, dtype=torch.float32)
    for i, f in enumerate(beats_list):
        beats_padded[i, :f.shape[0]] = f
        beats_mask[i, :f.shape[0]] = 1.0
    return {
        "waveform": waveforms,
        "beats_padded": beats_padded,
        "beats_mask": beats_mask,
        "beats_valid_ratio": torch.tensor([b["beats_valid_ratio"] for b in batch], dtype=torch.float32),
        "duration": torch.tensor([b["duration"] for b in batch], dtype=torch.float32),
        "coarse_target": torch.stack([b["coarse_target"] for b in batch]),
        "fine_clip_target": torch.stack([b["fine_clip_target"] for b in batch]),
        "event_clip_target": torch.stack([b["event_clip_target"] for b in batch]),
        "event_spans": [b["event_spans"] for b in batch],
        "tier": torch.tensor([b["tier"] for b in batch], dtype=torch.long),
        "src_idx": torch.tensor([b["src_idx"] for b in batch], dtype=torch.long),
        "full_idx": torch.tensor([b["full_idx"] for b in batch], dtype=torch.long),
    }


In [ ]:
# =============================================================================
# CELL: Model -- accepts beats_valid_ratio and gates fusion on valid region
# =============================================================================
class VaaniLiteSED(nn.Module):
    def __init__(self, beats_model=None, n_fine=1, n_coarse=7,
                 beats_dim=768, proj_dim=192, gru_hidden=128, gru_layers=2,
                 mel_frontend=None):
        super().__init__()
        self.beats = beats_model; self.mel_frontend = mel_frontend
        self.beats_dim = beats_dim; self.proj_dim = proj_dim
        self.dfd_lka = DFDLKAEncoder(channels=(32, 64, 128, 192))
        self.beats_proj = nn.Linear(beats_dim, proj_dim)
        self.fdy_proj = nn.Linear(192, proj_dim)
        self.fuse_gate = nn.Sequential(nn.Linear(proj_dim * 2, proj_dim), nn.Sigmoid())
        self.fuse_norm = nn.LayerNorm(proj_dim)
        self.gru = nn.GRU(proj_dim, gru_hidden, num_layers=gru_layers,
                          batch_first=True, bidirectional=True, dropout=0.10)
        gru_out = 2 * gru_hidden
        self.gru_norm = nn.LayerNorm(gru_out)
        self.temporal_refine = TemporalRefinement(gru_out)
        self.event_frame_head = nn.Linear(gru_out, 1)
        self.event_onset_head = nn.Linear(gru_out, 1)
        self.event_offset_head = nn.Linear(gru_out, 1)
        self.event_attn = nn.Conv1d(gru_out, 1, kernel_size=1)
        self.event_clip_proj = nn.Linear(gru_out, 1)
        self.fine_attn = nn.Conv1d(gru_out, n_fine, kernel_size=1)
        self.fine_clip_proj = nn.Linear(gru_out, n_fine)
        self.coarse_attn = nn.Conv1d(gru_out, n_coarse, kernel_size=1)
        self.coarse_proj = nn.Linear(gru_out, n_coarse)

    def _beats_features(self, waveform, beats_cached):
        if beats_cached is not None: return beats_cached
        if self.beats is None: raise RuntimeError("No cached BEATs and no live BEATs model.")
        with torch.no_grad():
            feat, _, _ = beats_extract_temporal(self.beats, waveform)
        return feat

    def forward(self, waveform, beats_cached=None, beats_mask=None,
                beats_valid_ratio=None, mel=None):
        if waveform.ndim == 3: waveform = waveform.squeeze(1)
        beats_feat = self._beats_features(waveform, beats_cached)
        beats_feat = self.beats_proj(beats_feat)
        if mel is None:
            if self.mel_frontend is None: raise RuntimeError("mel_frontend required when mel=None")
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        if beats_feat.shape[1] != T:
            beats_feat = F.interpolate(beats_feat.transpose(1, 2), size=T,
                                       mode="linear", align_corners=False).transpose(1, 2)
        gate = self.fuse_gate(torch.cat([beats_feat, fdy], dim=-1))
        # Gate fusion on the BEATs-valid region (short clips).
        if beats_valid_ratio is not None:
            arange = torch.arange(T, device=waveform.device, dtype=torch.float32).unsqueeze(0)
            valid_T = beats_valid_ratio.unsqueeze(1) * T
            frame_valid = (arange < valid_T).float()
            gate = gate * frame_valid.unsqueeze(-1)
        x = self.fuse_norm(gate * beats_feat + (1.0 - gate) * fdy)
        x, _ = self.gru(x); x = self.gru_norm(x); x = self.temporal_refine(x)
        event_frame_logits = self.event_frame_head(x)
        event_onset_logits = self.event_onset_head(x)
        event_offset_logits = self.event_offset_head(x)
        xf = x.transpose(1, 2)
        event_attn = F.softmax(self.event_attn(xf), dim=-1)
        event_pool = torch.einsum("bct,btd->bcd", event_attn, x)
        event_clip_logits = (event_pool * self.event_clip_proj.weight).sum(-1) + self.event_clip_proj.bias
        fine_attn = F.softmax(self.fine_attn(xf), dim=-1)
        coarse_attn = F.softmax(self.coarse_attn(xf), dim=-1)
        fine_pool = torch.einsum("bct,btd->bcd", fine_attn, x)
        coarse_pool = torch.einsum("bct,btd->bcd", coarse_attn, x)
        fine_clip_logits = torch.einsum("bcd,cd->bc", fine_pool, self.fine_clip_proj.weight) + self.fine_clip_proj.bias
        coarse_logits = torch.einsum("bcd,cd->bc", coarse_pool, self.coarse_proj.weight) + self.coarse_proj.bias
        return {"event_frame_logits": event_frame_logits,
                "event_onset_logits": event_onset_logits,
                "event_offset_logits": event_offset_logits,
                "event_clip_logits": event_clip_logits,
                "fine_clip_logits": fine_clip_logits,
                "coarse_logits": coarse_logits,
                "encoder_time_steps": T}


In [ ]:
# =============================================================================
# CELL: Losses (FIXED: AsymmetricFocalLoss pt was double-complemented for
# target=0, which inverted the focal weighting for negative frames --
# see explanation above. xs_neg is now used directly as pt for negatives,
# matching the canonical Asymmetric Loss formulation, instead of taking
# (1 - xs_neg) again.)
# =============================================================================
class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg; self.gamma_pos = gamma_pos; self.clip = clip
    def forward(self, logits, targets, mask=None):
        probs = torch.sigmoid(logits)
        xs_neg = 1.0 - probs
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        pt = probs * targets + xs_neg * (1.0 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * (1.0 - pt).pow(gamma)
        if mask is not None:
            m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
            return (loss * m).sum() / m.sum().clamp_min(1.0)
        return loss.mean()

def masked_bce(logits, targets, mask=None):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    if mask is None: return loss.mean()
    m = mask.unsqueeze(-1).expand_as(loss) if mask.ndim == loss.ndim - 1 else mask
    return (loss * m).sum() / m.sum().clamp_min(1.0)

def classwise_attention_bce(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)

def build_event_targets(event_spans_batch, batch_size, T, durations, device,
                        window_seconds=8.0, safe_margin_seconds=None):
    frame_target = torch.zeros(batch_size, T, 1, device=device)
    onset_target = torch.zeros(batch_size, T, 1, device=device)
    offset_target = torch.zeros(batch_size, T, 1, device=device)
    positive_mask = torch.zeros(batch_size, T, device=device)
    valid_mask = torch.zeros(batch_size, T, device=device)
    safe_negative_mask = torch.zeros(batch_size, T, device=device)
    centers = (torch.arange(T, device=device, dtype=torch.float32) + 0.5) * (window_seconds / T)
    for b in range(batch_size):
        dur = max(float(durations[b]), 1e-6)
        valid_mask[b] = (centers < float(durations[b])).float()
        spans = event_spans_batch[b]
        for start, end in spans:
            start = float(start); end = float(end)
            active = (centers >= start) & (centers < end)
            frame_target[b, active, 0] = 1.0; positive_mask[b, active] = 1.0
            sigma = max(dur / T, 0.03)
            on = torch.exp(-0.5 * ((centers - start) / sigma).pow(2))
            off = torch.exp(-0.5 * ((centers - end) / sigma).pow(2))
            onset_target[b, :, 0] = torch.maximum(onset_target[b, :, 0], on)
            offset_target[b, :, 0] = torch.maximum(offset_target[b, :, 0], off)
        if safe_margin_seconds is not None:
            if not spans:
                safe_negative_mask[b] = valid_mask[b]
            else:
                md = torch.full((T,), float("inf"), device=device)
                for start, end in spans:
                    start = float(start); end = float(end)
                    inside = (centers >= start) & (centers < end)
                    d = torch.minimum((centers - start).abs(), (centers - end).abs())
                    d = torch.where(inside, torch.zeros_like(d), d)
                    md = torch.minimum(md, d)
                safe_negative_mask[b] = valid_mask[b] * (md > safe_margin_seconds).float()
    return frame_target, onset_target, offset_target, valid_mask, positive_mask, safe_negative_mask

def consistency_loss(student_logits, teacher_logits, valid_mask, confidence_floor=0.60):
    s = torch.sigmoid(student_logits); t = torch.sigmoid(teacher_logits).detach()
    conf = torch.maximum(t, 1.0 - t)
    conf = ((conf - confidence_floor) / max(1.0 - confidence_floor, 1e-6)).clamp(0, 1)
    w = valid_mask.unsqueeze(-1) * conf
    return ((s - t).pow(2) * w).sum() / w.sum().clamp_min(1.0)

def hierarchical_consistency(fine_clip_logits, coarse_logits, fine_to_coarse_idx):
    fine_p = torch.sigmoid(fine_clip_logits); coarse_p = torch.sigmoid(coarse_logits)
    penalties = []
    for fi, ci in enumerate(fine_to_coarse_idx.tolist()):
        penalties.append(F.relu(fine_p[:, fi] - coarse_p[:, ci]).mean())
    return torch.stack(penalties).mean() if penalties else fine_clip_logits.new_tensor(0.0)


In [ ]:
# =============================================================================
# CELL: Mean teacher + decoder + scorer
# =============================================================================
@torch.no_grad()
def update_teacher(student, teacher, decay=0.999):
    for sp, tp in zip(student.parameters(), teacher.parameters()):
        if tp is sp: continue
        tp.data.mul_(decay).add_(sp.data, alpha=1.0 - decay)
    for sb, tb in zip(student.buffers(), teacher.buffers()):
        if tb is sb: continue
        tb.copy_(sb)

def freeze_model(model):
    for p in model.parameters(): p.requires_grad_(False)
    model.eval()

def smooth_probs(probs_1d, window):
    if window is None or window <= 1: return probs_1d
    k = int(window)
    if k % 2 == 0: k += 1
    if k >= len(probs_1d): k = len(probs_1d) - (1 - len(probs_1d) % 2)
    if k < 3: return probs_1d
    return medfilt(probs_1d, kernel_size=k)

def snap_to_peak(probs, idx, radius):
    if probs is None or radius <= 0: return idx
    lo = max(0, idx - radius); hi = min(len(probs), idx + radius + 1)
    if hi <= lo: return idx
    return lo + int(np.argmax(probs[lo:hi]))

def decode_events_single(frame_probs, fps, threshold=0.5, smooth_window=5,
                         min_duration=0.15, merge_gap=0.10,
                         onset_probs=None, offset_probs=None,
                         snap_radius_seconds=0.15):
    probs = smooth_probs(np.asarray(frame_probs, dtype=np.float32), smooth_window)
    binary = (probs >= threshold).astype(np.int32)
    raw = []; in_ev, s_idx = False, 0
    for i, v in enumerate(binary):
        if v and not in_ev: in_ev, s_idx = True, i
        elif not v and in_ev: in_ev = False; raw.append((s_idx, i))
    if in_ev: raw.append((s_idx, len(binary)))
    radius = max(1, int(round(snap_radius_seconds * fps)))
    snapped = []
    for s, e in raw:
        ss = snap_to_peak(onset_probs, s, radius)
        es = snap_to_peak(offset_probs, e, radius)
        if es <= ss: ss, es = s, e
        snapped.append((ss, es))
    merged = []
    for s, e in snapped:
        if merged and (s - merged[-1][1]) / fps <= merge_gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else: merged.append((s, e))
    out = []
    for s, e in merged:
        onset, offset = s / fps, e / fps
        if offset - onset >= min_duration: out.append((float(onset), float(offset)))
    return out

def match_events(ref_events, pred_events, tolerance_frac=0.20, min_tol=0.05):
    mr, mp = set(), set(); cand = []
    for ri, (ro, rf) in enumerate(ref_events):
        tol = max(tolerance_frac * (rf - ro), min_tol)
        for pi, (po, pf) in enumerate(pred_events):
            if abs(po - ro) <= tol and abs(pf - rf) <= tol:
                cand.append((abs(po - ro) + abs(pf - rf), ri, pi))
    for _, ri, pi in sorted(cand):
        if ri not in mr and pi not in mp: mr.add(ri); mp.add(pi)
    tp = len(mr)
    return tp, len(pred_events) - tp, len(ref_events) - tp

def event_based_f1(ref_data, pred_data):
    TP = FP = FN = 0
    for cid, re in ref_data.items():
        tp, fp, fn = match_events(re, pred_data.get(cid, []))
        TP += tp; FP += fp; FN += fn
    for cid in pred_data:
        if cid not in ref_data: FP += len(pred_data[cid])
    p = TP / (TP + FP) if (TP + FP) else 0.0
    r = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return f1, p, r

def events_to_frames(events, max_time, frame_len=0.01):
    n = int(max_time / frame_len) + 1
    m = np.zeros(n, dtype=np.int32)
    for on, off in events:
        i0 = int(on / frame_len); i1 = min(int(off / frame_len) + 1, n)
        if i1 > i0: m[i0:i1] = 1
    return m

def segment_dice(ref_data, pred_data):
    scores = []
    for cid, re in ref_data.items():
        pe = pred_data.get(cid, []); all_ev = re + pe
        if not all_ev: scores.append(1.0); continue
        mt = max(off for _, off in all_ev) + 0.5
        r = events_to_frames(re, mt); p = events_to_frames(pe, mt)
        i = int((r & p).sum()); t = int(r.sum() + p.sum())
        scores.append(1.0 if t == 0 else 2.0 * i / t)
    return sum(scores) / len(scores) if scores else 0.0

def official_combined_score(ref_data, pred_data):
    f1, p, r = event_based_f1(ref_data, pred_data)
    dice = segment_dice(ref_data, pred_data)
    return {"event_f1": f1, "precision": p, "recall": r,
            "segment_dice": dice, "combined": f1 + dice}


In [ ]:
# =============================================================================
# CELL: Split / label space / priors (index-aware)
# =============================================================================
def split_train_val(dataset, val_fraction=0.10, seed=42, group_col="district"):
    groups = dataset[group_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    ia = np.arange(len(dataset))
    tr, va = next(gss.split(ia, groups=groups))
    return tr.tolist(), va.tolist()

def build_label_space(ds, indices=None, min_tag_count_to_keep=5):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    tag_counts = Counter(); tag_to_coarse_votes = defaultdict(Counter)
    try:
        subcat_col = ds["NoiseSubCategoryTimeStamp"]; cats_col = ds["NoiseCategory"]
        use_cols = True
    except Exception:
        use_cols = False
    for i in idxs:
        if use_cols:
            events = subcat_col[i] or []; cats = cats_col[i] or []
        else:
            s = ds[i]; events = s.get("NoiseSubCategoryTimeStamp") or []; cats = s.get("NoiseCategory") or []
        for e in events:
            raw = e.get("tag", "")
            if not raw: continue
            tag_counts[raw] += 1
            for c in cats:
                if c in COARSE_TO_IDX: tag_to_coarse_votes[raw][c] += 1
    raw_to_canonical, cluster_report = build_tag_clusters(tag_counts)
    canon_counts = Counter(); canon_coarse_votes = defaultdict(Counter)
    for raw, cnt in tag_counts.items():
        canon = raw_to_canonical[raw]; canon_counts[canon] += cnt
        for c, v in tag_to_coarse_votes[raw].items(): canon_coarse_votes[canon][c] += v
    canonical_tags = sorted(t for t, c in canon_counts.items() if c >= min_tag_count_to_keep)
    f2c = torch.zeros(len(canonical_tags), dtype=torch.long)
    for i, tag in enumerate(canonical_tags):
        v = canon_coarse_votes.get(tag)
        f2c[i] = COARSE_TO_IDX[v.most_common(1)[0][0]] if v else 0
    return canonical_tags, raw_to_canonical, f2c, cluster_report

def estimate_global_duration_prior(ds, indices=None):
    idxs = list(range(len(ds))) if indices is None else list(indices)
    try:
        qcol = ds["annotationQuality"]; scol = ds["NoiseSubCategoryTimeStamp"]; use_cols = True
    except Exception:
        use_cols = False
    out = []
    for i in idxs:
        if use_cols:
            if qcol[i] != "verified_timestamps": continue
            events = scol[i] or []
        else:
            s = ds[i]
            if s.get("annotationQuality") != "verified_timestamps": continue
            events = s.get("NoiseSubCategoryTimeStamp") or []
        for e in events:
            try: d = float(e["end"]) - float(e["start"])
            except (TypeError, ValueError, KeyError): continue
            if d > 0: out.append(d)
    return float(np.median(out)) if out else 1.0

def compute_adaptive_window(median_duration, fps, min_window=3, max_window=25):
    w = int(round(median_duration * fps * 0.5))
    w = max(min_window, min(max_window, w))
    return w + 1 if w % 2 == 0 else w

def weak_augment(mel): return apply_mel_specaugment(mel, 1, 1, 0.05, 8)
def strong_augment(mel): return apply_mel_specaugment(mel, 2, 2, 0.12, 16)

# =============================================================================
# TIER_FRAME_SCALE -- per-tier scale applied ONLY to the frame-level losses
# (event_frame, onset, offset).
#
# Silver (unverified_timestamps) carries a trustworthy clip category but NOT
# trustworthy start/end times -- its spans tile ~68% of every clip, so as
# frame-level supervision it says "most of the audio is an event", which is
# false. The previous full 33-epoch run confirmed this: with Silver in the
# frame losses, frac_pos_frame pinned at ~0.60, exactly Silver's positive
# frame rate, and Event-F1 never left ~0.07 across all three stages.
#
# Fix: zero Silver out of the frame/onset/offset losses. Gold is the only
# tier whose timestamps can define per-frame positives and negatives.
# Silver still drives the clip-level losses (fine_clip, coarse, event_clip)
# below -- those only need the clip category, which Silver has.
#
# TIER_PSEUDO_BRONZE stays at 0.4 because Stage 3 pseudo-labels are generated
# by the model's own Gold-trained decoder.
# =============================================================================
TIER_FRAME_SCALE = {TIER_VERIFIED: 1.0, TIER_UNVERIFIED: 0.0,
                    TIER_WEAK_ONLY: 0.0, TIER_PSEUDO_BRONZE: 0.4}

# Alias kept so any external references to TIER_LOSS_SCALE still resolve.
TIER_LOSS_SCALE = TIER_FRAME_SCALE

def make_tier_scales(tier_tensor, device, scale_map=None):
    scale_map = TIER_FRAME_SCALE if scale_map is None else scale_map
    scale = torch.zeros(tier_tensor.shape[0], device=device)
    for t, s in scale_map.items(): scale[tier_tensor == t] = s
    return scale


In [ ]:
# =============================================================================
# CELL: Inference utilities -- strict cache lookup
# =============================================================================
@torch.no_grad()
def get_model_time_grid(model, device, window_seconds, sample_rate):
    model.eval()
    dummy = torch.zeros(1, int(round(window_seconds * sample_rate)), device=device)
    out = model(dummy)
    T = out["encoder_time_steps"]
    return T, T / window_seconds

def _ref_events_for_sample(sample):
    evs = []
    for e in (sample.get("NoiseSubCategoryTimeStamp") or []):
        try: s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError): continue
        if en > s: evs.append((s, en))
    return evs

def _window_starts(total_samples, win, stride):
    if total_samples <= win: return [0]
    starts = list(range(0, total_samples - win + 1, stride))
    if starts[-1] + win < total_samples: starts.append(total_samples - win)
    return starts

@torch.no_grad()
def _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                            window_seconds, stride_seconds,
                            beats_full_np=None, full_duration=None):
    total = wav.numel(); duration_sec = total / sample_rate
    win = int(round(window_seconds * sample_rate)); stride = int(round(stride_seconds * sample_rate))
    total_frames = int(round(duration_sec * fps)) + T_win + 1
    sums = {k: np.zeros(total_frames, dtype=np.float64) for k in ("frame", "onset", "offset")}
    wsum = np.zeros(total_frames, dtype=np.float64)
    for s0 in _window_starts(total, win, stride):
        chunk = wav[s0:s0+win]
        pad_len = win - chunk.numel()
        if pad_len > 0: chunk = F.pad(chunk, (0, pad_len))
        chunk = chunk.unsqueeze(0).to(device)
        beats_cached = None
        if beats_full_np is not None and full_duration is not None:
            T_full = beats_full_np.shape[0]
            win_start_s = s0 / sample_rate
            win_end_s   = (s0 + win) / sample_rate
            # Use the same patch-time mapping as the dataset.
            p0 = int(np.floor((win_start_s - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
            p1 = int(np.ceil((win_end_s   - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
            p0 = max(0, min(p0, T_full - 1)); p1 = max(p0 + 1, min(p1, T_full))
            beats_cached = torch.from_numpy(beats_full_np[p0:p1]).float().unsqueeze(0).to(device)
        out = model(chunk, beats_cached=beats_cached)
        frame_l = out["event_frame_logits"][0, :, 0].detach().cpu().numpy()
        onset_l = out["event_onset_logits"][0, :, 0].detach().cpu().numpy()
        offset_l = out["event_offset_logits"][0, :, 0].detach().cpu().numpy()
        off = int(round((s0 / sample_rate) * fps))
        end = min(off + T_win, total_frames); L = end - off
        if L > 0:
            sums["frame"][off:end] += frame_l[:L]
            sums["onset"][off:end] += onset_l[:L]
            sums["offset"][off:end] += offset_l[:L]
            wsum[off:end] += 1.0
    valid_len = min(int(round(duration_sec * fps)) + 1, total_frames)
    w = np.maximum(wsum[:valid_len], 1e-6)
    merged = {k: sums[k][:valid_len] / w for k in sums}
    return {k: 1.0 / (1.0 + np.exp(-merged[k])) for k in merged}

@torch.no_grad()
def compute_val_probabilities(model, parent_hf, full_indices, device, sample_rate,
                              beats_cache_dir=None, wav_cache_dir=None, label_table=None,
                              window_seconds=8.0, stride_seconds=4.0, max_clips=None):
    model.eval()
    T_win, fps = get_model_time_grid(model, device, window_seconds, sample_rate)
    n = len(full_indices) if max_clips is None else min(max_clips, len(full_indices))
    ref_data, probs_by_clip = {}, {}
    for i in range(n):
        full_idx = int(full_indices[i])
        # CHANGED: cache-aware waveform load (see _load_waveform) instead of
        # a live decode+resample every epoch this runs.
        wav = _load_waveform(full_idx, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        beats_full_np = None
        if beats_cache_dir is not None:
            p = resolve_cache_path(full_idx, beats_cache_dir)
            if p is None:
                raise RuntimeError(f"Missing cache for full_idx={full_idx} during val inference "
                                   f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
            beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        # CHANGED: label_table avoids the row access that would otherwise
        # decode audio just to read the clip id / ref events.
        if label_table is not None:
            cid = str(label_table["id"][full_idx] or full_idx)
            ref_data[cid] = _ref_events_for_full_idx(full_idx, label_table)
        else:
            sample = parent_hf[full_idx]
            cid = str(sample.get("id") or sample.get("clip_id") or full_idx)
            ref_data[cid] = _ref_events_for_sample(sample)
        probs_by_clip[cid] = merged
    return ref_data, probs_by_clip, fps

def _decode_clip(cached, fps, thr, sw, md, mg, sr):
    return decode_events_single(cached["frame"], fps, threshold=thr, smooth_window=sw,
                                min_duration=md, merge_gap=mg,
                                onset_probs=cached["onset"], offset_probs=cached["offset"],
                                snap_radius_seconds=sr)

def grid_search_decode_params(ref_data, probs_by_clip, fps, median_duration,
                              thresholds=(0.3, 0.4, 0.5, 0.6, 0.7),
                              smooth_windows=None,
                              min_duration_fracs=(0.1, 0.2, 0.3),
                              merge_gaps=(0.05, 0.10, 0.15, 0.20),
                              snap_radius_seconds_options=(0.10, 0.25)):
    if smooth_windows is None:
        base_w = compute_adaptive_window(median_duration, fps)
        smooth_windows = sorted(set([max(3, base_w // 2), base_w, base_w * 2]))
    best = {"combined": -1.0}
    for thr in thresholds:
        for sw in smooth_windows:
            for mdf in min_duration_fracs:
                md = max(0.05, mdf * median_duration)
                for g in merge_gaps:
                    for sr in snap_radius_seconds_options:
                        pred = {cid: _decode_clip(c, fps, thr, sw, md, g, sr)
                                for cid, c in probs_by_clip.items()}
                        score = official_combined_score(ref_data, pred)
                        if score["combined"] > best["combined"]:
                            best = {**score, "threshold": thr, "smooth_window": sw,
                                    "min_duration": md, "merge_gap": g,
                                    "snap_radius_seconds": sr}
    return best

@torch.no_grad()
def generate_pseudo_labels(teacher_model, parent_hf, bronze_full_indices, sample_rate, device,
                           decode_params, beats_cache_dir=None, wav_cache_dir=None,
                           window_seconds=8.0, stride_seconds=4.0,
                           min_confidence=0.65, max_clips=None):
    teacher_model.eval()
    T_win, fps = get_model_time_grid(teacher_model, device, window_seconds, sample_rate)
    indices = [int(i) for i in bronze_full_indices]
    if max_clips is not None: indices = indices[:max_clips]
    overrides = {}
    for fi in indices:
        # CHANGED: cache-aware waveform load instead of a live decode --
        # this only runs once per Bronze clip per stage, so lower total
        # impact than the training loader, but still free to fix the same way.
        wav = _load_waveform(fi, parent_hf, sample_rate, wav_cache_dir)
        full_dur = wav.numel() / sample_rate
        p = resolve_cache_path(fi, beats_cache_dir)
        if p is None:
            raise RuntimeError(f"Missing cache for full_idx={fi} during pseudo-labeling "
                               f"(checked {_normalize_cache_dirs(beats_cache_dir)})")
        beats_full_np = np.load(p).astype(np.float32)
        merged = _windowed_logit_tracks(teacher_model, wav, sample_rate, device, T_win, fps,
                                        window_seconds, stride_seconds,
                                        beats_full_np=beats_full_np, full_duration=full_dur)
        spans = decode_events_single(
            merged["frame"], fps, threshold=decode_params["threshold"],
            smooth_window=decode_params["smooth_window"],
            min_duration=decode_params["min_duration"], merge_gap=decode_params["merge_gap"],
            onset_probs=merged["onset"], offset_probs=merged["offset"],
            snap_radius_seconds=decode_params.get("snap_radius_seconds", 0.15))
        accepted = []
        for s, e in spans:
            si = int(round(s * fps)); ei = min(int(round(e * fps)), len(merged["frame"]) - 1)
            peak = float(merged["frame"][si:ei+1].max()) if ei >= si else 0.0
            if peak >= min_confidence: accepted.append({"start": s, "end": e})
        if accepted: overrides[fi] = {"events": accepted}
    return overrides


In [ ]:
# =============================================================================
# CELL: Config
# =============================================================================
CONFIG = dict(
    seed=42, val_fraction=0.10, sample_rate=16000, crop_seconds=8.0, n_mels=128,
    min_tag_count_to_keep=5, batch_size=8, grad_accum_steps=2,
    # num_workers=0: confirmed via a live AttributeError traceback that Windows'
    # spawn-based multiprocessing can't unpickle VaaniNoiseDataset in a worker
    # process, since it's defined in a notebook cell, not an importable .py
    # module -- num_workers>0 deadlocks silently on this setup.
    num_workers=0,
    stage1_epochs=15, stage2_epochs=10, stage3_epochs=8,
    lr_stage1=3e-4, lr_stage2=8e-5, lr_stage3=5e-5,
    lr_warmup_steps=500, teacher_decay=0.999,
    # silver_negative_weight is INERT for Silver now: TIER_FRAME_SCALE
    # [TIER_UNVERIFIED] = 0 zeros Silver out of the frame-level losses
    # entirely before this weight is ever applied. Kept at 0.75 so that if
    # you later re-admit Silver partially, the existing negative-frame
    # weighting still applies unchanged.
    silver_negative_weight=0.75,
    # gamma_neg_frame: passed into AsymmetricFocalLoss in run_epoch.
    # Was the class default 4.0. With Silver removed from the frame losses,
    # Gold is the only source of frame-level negatives left -- and
    # gamma_neg=4.0 was down-weighting those too aggressively on a class
    # balance that isn't actually extreme. 2.0 keeps a mild positive/
    # negative asymmetry without starving the head of "not event" gradient.
    # Watch frac_pos_frame in progress_live.txt: it should settle around
    # 0.15-0.35. If it drifts above 0.55, drop this to 1.0 and leave Silver
    # at 0 -- do NOT re-enable Silver to compensate.
    gamma_neg_frame=2.0,
    # gold_target_fraction: expected share of Gold rows per training batch
    # under make_gold_aware_sampler (see train_one_stage). Gold is only
    # ~9-12% of any stage's subset, so plain uniform shuffling at
    # batch_size=8 leaves 36-46% of batches with ZERO Gold rows -- and
    # since frame/onset/offset losses only get gradient from Gold rows
    # (TIER_FRAME_SCALE), those heads get no update at all on that
    # fraction of steps. 0.35 drops that to roughly 3%.
    gold_target_fraction=0.35,
    max_eval_clips=300,
)


## Part 2 — V5 core definitions


In [ ]:
# =============================================================================
# V5 CORE DEFINITIONS (no work is done in this cell; it only defines things)
#
#  1. time alignment: BEATs rows <-> 8 s windows on the 0.16 s patch grid
#  2. labels: per-event category, per-crop weak targets with "unsure" masks
#  3. VaaniSEDV5: your V3 network + 7 category frame tracks + MIL weak head
#  4. dataset (natural + synthetic rows), tier sampler, collate
#  5. targets + losses
#  6. windowed inference -> per-category tracks
#  7. per-category decoder + flatten
# =============================================================================
import math, time, json, pickle, hashlib, zipfile, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast, GradScaler

SR = 16000
W_SEC = 8.0
WIN_SAMPLES = int(W_SEC * SR)                      # 128000
PATCH_SAMPLES = int(round(BEATS_PATCH_HOP_SEC * SR))  # 2560 = one BEATs patch (0.16 s)
STRIDE_SAMPLES = WIN_SAMPLES // 2                  # 4 s = exactly 25 patches
CANON_T = int(round(W_SEC / BEATS_PATCH_HOP_SEC))  # 50 BEATs rows per window
assert STRIDE_SAMPLES % PATCH_SAMPLES == 0 and WIN_SAMPLES % PATCH_SAMPLES == 0

CATS = list(COARSE_CATEGORIES)
NC = len(CATS)
CAT_IDX = {c: i for i, c in enumerate(CATS)}
SHORT_CATS = {"human_non_speech", "phone_signal_alarm"}     # decoder parameter groups
CAT_GROUP = ["short" if c in SHORT_CATS else "long" for c in CATS]

T_GOLD, T_SILVER, T_BRONZE, T_SYNTH = 0, 1, 2, 4
TIER_NAME = {T_GOLD: "gold", T_SILVER: "silver", T_BRONZE: "bronze", T_SYNTH: "synth"}
BAD_INDICES = {3561, 48940}                        # known corrupt clips, no cache entry

# BEATs patch k covers fbank frames 16k..16k+15 (25 ms window, 10 ms hop), so its
# centre is at 0.0125 + 0.01*(16k+7.5) = 0.0875 + 0.16k s. Linear interpolation of
# 50 rows onto the 801 mel frames puts row j at w + 0.08 + 0.16j for a window that
# starts at w. Matching the two gives k0 = (w - 0.0075)/0.16; with every window start
# on the 2560-sample grid this is exactly k0 = w/0.16 with a 7.5 ms residual.
PATCH_CENTRE_OFFSET = 0.0875
ALIGN_TOL_SEC = 0.0076


def beats_T_expected(n_samples):
    """Rows BEATs produces for an n-sample clip (kaldi fbank, snip_edges, 16-frame patches)."""
    frames = 1 + (int(n_samples) - 400) // 160 if n_samples >= 400 else 0
    return max(0, frames // 16)


def beats_rows_for_window(beats_full, start_sample):
    """BEATs rows for the window starting at start_sample (must be on the patch grid).
    Returns (rows[CANON_T, D], n_real_rows)."""
    assert start_sample % PATCH_SAMPLES == 0, f"window start {start_sample} is off the patch grid"
    w = start_sample / SR
    k0 = int(round((w - (PATCH_CENTRE_OFFSET - 0.08)) / BEATS_PATCH_HOP_SEC))
    k0 = max(0, k0)
    # assertion: first row centre lands where interpolation will put it
    assert abs((PATCH_CENTRE_OFFSET + BEATS_PATCH_HOP_SEC * k0) - (w + 0.08)) <= ALIGN_TOL_SEC + 1e-9
    rows = beats_full[k0:k0 + CANON_T]
    n_real = rows.shape[0]
    if n_real < CANON_T:
        rows = np.concatenate([rows, np.zeros((CANON_T - n_real, beats_full.shape[1]), rows.dtype)], 0)
    return rows, n_real


def grid_window_starts(total_samples):
    """8 s windows, 4 s stride, every start on the 0.16 s patch grid. The last window
    may run past the clip end by < 0.16 s (zero-padded, gated off)."""
    if total_samples <= WIN_SAMPLES:
        return [0]
    starts = list(range(0, total_samples - WIN_SAMPLES + 1, STRIDE_SAMPLES))
    last = int(math.ceil((total_samples - WIN_SAMPLES) / PATCH_SAMPLES)) * PATCH_SAMPLES
    if starts[-1] < last:
        starts.append(last)
    return starts


# ---------------------------------------------------------------------------
# 2. labels
# ---------------------------------------------------------------------------
_TAG_TO_CAT = {}   # filled by build_tag_to_cat (fallback only; events carry 'category')


def build_tag_to_cat(label_table, indices):
    votes = defaultdict(Counter)
    for i in indices:
        for e in (label_table["NoiseSubCategoryTimeStamp"][i] or []):
            c = e.get("category") if isinstance(e, dict) else None
            if c in CAT_IDX:
                votes[e.get("tag", "")][c] += 1
    _TAG_TO_CAT.clear()
    _TAG_TO_CAT.update({t: v.most_common(1)[0][0] for t, v in votes.items()})
    return _TAG_TO_CAT


def parse_events(raw_events, clip_cats=None):
    """-> list of (start, end, cat_idx). Category from the event, else its tag, else
    the clip's only category. Events whose category cannot be resolved are dropped
    (counted by the caller)."""
    out = []
    for e in (raw_events or []):
        try:
            s = float(e["start"]); en = float(e["end"])
        except (TypeError, ValueError, KeyError):
            continue
        if not en > s:
            continue
        c = e.get("category")
        if c not in CAT_IDX:
            c = _TAG_TO_CAT.get(e.get("tag", ""))
        if c not in CAT_IDX and clip_cats and len(clip_cats) == 1:
            c = clip_cats[0]
        if c in CAT_IDX:
            out.append((s, en, CAT_IDX[c]))
    return out


# ---------------------------------------------------------------------------
# 3. model
# ---------------------------------------------------------------------------
UNUSED_V3_HEADS = ("event_frame_head", "event_attn", "event_clip_proj", "fine_attn",
                   "fine_clip_proj", "coarse_attn", "coarse_proj")


class VaaniSEDV5(VaaniLiteSED):
    """Your V3 network (same mel branch, BEATs projection, gated fusion, BiGRU,
    TemporalRefinement, onset/offset heads) with
      * cat_frame_head: Linear(256 -> 7), one frame track per category
      * cat_attn: Conv1d(256 -> 7) attention that pools the *frame-track
        probabilities* into clip-level category probabilities (MIL), so weak labels
        train the same tracks the decoder reads.
    BEATs is never run inside this module (beats_model=None): features always come
    from a cache, so a missing cache entry can never silently become live BEATs."""

    def __init__(self, n_fine, beats_dim=768, mel_frontend=None, feat_dropout=0.2):
        super().__init__(beats_model=None, n_fine=n_fine, n_coarse=len(COARSE_CATEGORIES),
                         beats_dim=beats_dim, mel_frontend=mel_frontend)
        gru_out = self.event_frame_head.in_features
        self.feat_drop = nn.Dropout(feat_dropout)
        self.cat_frame_head = nn.Linear(gru_out, NC)
        self.cat_attn = nn.Conv1d(gru_out, NC, kernel_size=1)

    def init_from_v3(self, v3_state):
        sd = {k: v for k, v in v3_state.items() if not k.startswith("beats.")}
        missing, unexpected = self.load_state_dict(sd, strict=False)
        assert set(missing) == {"cat_frame_head.weight", "cat_frame_head.bias",
                                "cat_attn.weight", "cat_attn.bias"}, f"unexpected missing keys {missing}"
        assert not unexpected, f"unexpected keys {unexpected}"
        with torch.no_grad():
            self.cat_frame_head.weight.copy_(self.event_frame_head.weight.expand(NC, -1))
            self.cat_frame_head.bias.copy_(self.event_frame_head.bias.expand(NC))
            self.cat_attn.weight.copy_(self.coarse_attn.weight)
            self.cat_attn.bias.copy_(self.coarse_attn.bias)

    def trainable_parameters(self):
        return [p for n, p in self.named_parameters()
                if p.requires_grad and not n.split(".")[0] in UNUSED_V3_HEADS]

    def forward(self, waveform, beats, gate_span, mel=None):
        """waveform (B, 128000); beats (B, 50, 768); gate_span (B, 2) = [lo, hi) in
        seconds inside the window where BEATs features are real."""
        if waveform.ndim == 3:
            waveform = waveform.squeeze(1)
        bf = self.beats_proj(beats)
        if mel is None:
            mel = self.mel_frontend(waveform)
        fdy = self.fdy_proj(self.dfd_lka(mel.unsqueeze(1)))
        T = fdy.shape[1]
        bf = F.interpolate(bf.transpose(1, 2), size=T, mode="linear", align_corners=False).transpose(1, 2)
        bf = self.feat_drop(bf)
        fdy = self.feat_drop(fdy)
        gate = self.fuse_gate(torch.cat([bf, fdy], dim=-1))
        centres = (torch.arange(T, device=waveform.device, dtype=torch.float32) + 0.5) * (W_SEC / T)
        gspan = gate_span.float()
        bvalid = (centres[None] >= gspan[:, :1]) & (centres[None] < gspan[:, 1:2])
        gate = gate * bvalid.to(gate.dtype).unsqueeze(-1)
        x = self.fuse_norm(gate * bf + (1.0 - gate) * fdy)
        x, _ = self.gru(x)
        x = self.temporal_refine(self.gru_norm(x))
        return {"cat_logits": self.cat_frame_head(x),                       # (B,T,7)
                "onset_logits": self.event_onset_head(x)[..., 0],           # (B,T)
                "offset_logits": self.event_offset_head(x)[..., 0],
                "attn_logits": self.cat_attn(x.transpose(1, 2)).transpose(1, 2),  # (B,T,7)
                "T": T}


def mil_pool(cat_logits, attn_logits, valid):
    """Clip-level category probability = attention-weighted mean of frame
    probabilities over valid frames."""
    p = torch.sigmoid(cat_logits.float())
    a = attn_logits.float().masked_fill(~valid.bool().unsqueeze(-1), -1e4)
    a = torch.softmax(a, dim=1)
    return (a * p).sum(1).clamp(1e-6, 1 - 1e-6)


def state_hash(sd):
    h = hashlib.sha1()
    for k in sorted(sd):
        h.update(k.encode()); h.update(sd[k].detach().float().cpu().numpy().tobytes()[:4096])
    return h.hexdigest()[:12]


# ---------------------------------------------------------------------------
# 4. data
# ---------------------------------------------------------------------------
class SynthStore:
    """Synthetic mixtures: <root>/wav/<id>.flac, <root>/beats/<id>.npy,
    <root>/manifest.jsonl. Separate ID namespace (s000000, v000000) -- never touches
    the global-index BEATs cache."""
    def __init__(self, root):
        self.root = root
        self.items = {}
        mp = os.path.join(root, "manifest.jsonl")
        if os.path.exists(mp):
            with open(mp, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        r = json.loads(line); self.items[r["id"]] = r

    def wav(self, sid):
        x, sr = sf.read(os.path.join(self.root, "wav", f"{sid}.flac"), dtype="float32")
        assert sr == SR
        return torch.from_numpy(np.ascontiguousarray(x))

    def beats(self, sid):
        p = os.path.join(self.root, "beats", f"{sid}.npy")
        if not os.path.exists(p):
            raise RuntimeError(f"missing synthetic BEATs entry {p} -- rebuild the synthetic cache")
        return np.load(p).astype(np.float32)


def load_nat_beats(gidx):
    p = resolve_cache_path(gidx, BEATS_CACHE_DIRS)
    if p is None:
        raise RuntimeError(f"Missing BEATs cache entry for global index {gidx} "
                           f"(checked {BEATS_CACHE_DIRS}). No live fallback by design.")
    assert os.path.basename(p) == f"{int(gidx):06d}.npy", (p, gidx)   # global_index == cache_index
    return np.load(p).astype(np.float32)


def check_beats_len(beats_full, n_samples, key):
    exp = beats_T_expected(n_samples)
    if abs(beats_full.shape[0] - exp) > 1:
        raise RuntimeError(f"BEATs rows {beats_full.shape[0]} != expected {exp} for {key} "
                           f"({n_samples / SR:.2f}s audio) -- cache/index misalignment?")


class V5Dataset(Dataset):
    """Rows are ("nat", global_index, tier) or ("syn", synth_id, T_SYNTH)."""

    def __init__(self, rows, full_dataset, label_table, synth_store=None, training=True,
                 shift_prob=0.8, beats_mask_prob=0.5, beats_mask_max=3,
                 bg_crop_prob=0.3, jitter=1.5):
        self.rows = list(rows)
        self.fd = full_dataset; self.lt = label_table; self.syn = synth_store
        self.training = training
        self.shift_prob = shift_prob; self.beats_mask_prob = beats_mask_prob
        self.beats_mask_max = beats_mask_max; self.bg_crop_prob = bg_crop_prob; self.jitter = jitter

    def __len__(self):
        return len(self.rows)

    def _load(self, kind, key):
        if kind == "nat":
            wav = _load_waveform(key, self.fd, SR, WAV_CACHE_DIRS)
            bf = load_nat_beats(key)
            cats = [c for c in (self.lt["NoiseCategory"][key] or []) if c in CAT_IDX]
            evs = parse_events(self.lt["NoiseSubCategoryTimeStamp"][key], cats)
        else:
            r = self.syn.items[key]
            wav = self.syn.wav(key); bf = self.syn.beats(key)
            evs = [(float(s), float(e), CAT_IDX[c]) for s, e, c in r["events"]]
            cats = sorted({c for _, _, c in r["events"]})
        check_beats_len(bf, wav.numel(), f"{kind}:{key}")
        return wav, bf, evs, cats

    def _mask_beats(self, rows):
        if random.random() >= self.beats_mask_prob:
            return rows
        rows = rows.copy()
        for _ in range(random.randint(1, 2)):
            w = random.randint(1, self.beats_mask_max)
            s = random.randint(0, max(0, rows.shape[0] - w))
            rows[s:s + w] = 0
        return rows

    def __getitem__(self, i):
        kind, key, tier = self.rows[i]
        wav, bf, evs, cats = self._load(kind, key)
        n = wav.numel()
        if n <= WIN_SAMPLES:
            k_max = (WIN_SAMPLES - n) // PATCH_SAMPLES
            k = random.randint(0, k_max) if (self.training and k_max > 0 and random.random() < self.shift_prob) else 0
            shift = k * PATCH_SAMPLES
            chunk = F.pad(wav, (shift, WIN_SAMPLES - n - shift))
            rows = bf[:CANON_T - k]
            n_real = rows.shape[0]
            rows = np.concatenate([np.zeros((k, bf.shape[1]), np.float32), rows,
                                   np.zeros((CANON_T - k - n_real, bf.shape[1]), np.float32)], 0)
            vs = shift / SR; ve = (shift + n) / SR
            ghi = min(ve, (k + n_real) * BEATS_PATCH_HOP_SEC)
            crop_evs = [(s + vs, e + vs, c) for s, e, c in evs]
            whole = True
        else:
            k_max = (n - WIN_SAMPLES) // PATCH_SAMPLES
            if evs and random.random() >= self.bg_crop_prob:
                s0, e0, _ = random.choice(evs)
                centre = 0.5 * (s0 + e0) + random.uniform(-self.jitter, self.jitter)
                k = int(round((centre * SR - WIN_SAMPLES / 2) / PATCH_SAMPLES))
            else:
                k = random.randint(0, k_max)
            k = max(0, min(k_max, k))
            start = k * PATCH_SAMPLES
            chunk = wav[start:start + WIN_SAMPLES]
            rows, n_real = beats_rows_for_window(bf, start)
            cs = start / SR
            vs, ve = 0.0, W_SEC
            ghi = min(ve, n_real * BEATS_PATCH_HOP_SEC)
            crop_evs = []
            for s, e, c in evs:
                ns, ne = max(0.0, s - cs), min(W_SEC, e - cs)
                if ne > ns:
                    crop_evs.append((ns, ne, c))
            whole = False
        if self.training:
            rows = self._mask_beats(rows)

        # weak (clip-level) targets z and per-category confidence mask m
        z = np.zeros(NC, np.float32); m = np.ones(NC, np.float32)
        in_crop = {c for s, e, c in crop_evs if (e - s) >= 0.05}
        if tier in (T_GOLD, T_SYNTH):
            for c in in_crop: z[c] = 1.0
        elif tier == T_SILVER:
            clip_c = {CAT_IDX[c] for c in cats} | {c for _, _, c in evs}
            if whole:
                for c in clip_c: z[c] = 1.0
            else:
                strong_in = {c for s, e, c in crop_evs if (e - s) >= 0.3}
                for c in clip_c:
                    if c in strong_in: z[c] = 1.0
                    else: m[c] = 0.0             # present in clip, unsure in this crop
        else:  # bronze: clip categories only
            clip_c = {CAT_IDX[c] for c in cats}
            for c in clip_c:
                if whole: z[c] = 1.0
                else: m[c] = 0.0
        return {"waveform": chunk.float(), "beats": torch.from_numpy(np.ascontiguousarray(rows, np.float32)),
                "vspan": (vs, ve), "gspan": (vs, ghi), "events": crop_evs,
                "z": torch.from_numpy(z), "zmask": torch.from_numpy(m), "tier": tier, "key": f"{kind}:{key}"}


def collate_v5(batch):
    return {"waveform": torch.stack([b["waveform"] for b in batch]),
            "beats": torch.stack([b["beats"] for b in batch]),
            "vspan": torch.tensor([b["vspan"] for b in batch], dtype=torch.float32),
            "gspan": torch.tensor([b["gspan"] for b in batch], dtype=torch.float32),
            "events": [b["events"] for b in batch],
            "z": torch.stack([b["z"] for b in batch]), "zmask": torch.stack([b["zmask"] for b in batch]),
            "tier": torch.tensor([b["tier"] for b in batch], dtype=torch.long),
            "key": [b["key"] for b in batch]}


def make_tier_sampler(rows, shares, num_samples):
    counts = Counter(r[2] for r in rows)
    live = {t: s for t, s in shares.items() if counts.get(t, 0) > 0 and s > 0}
    tot = sum(live.values())
    print("  sampler target shares:", {TIER_NAME[t]: round(s / tot, 3) for t, s in live.items()},
          "| rows:", {TIER_NAME[t]: counts.get(t, 0) for t in shares})
    w = torch.tensor([live.get(r[2], 0.0) / tot / counts[r[2]] if r[2] in live else 0.0 for r in rows],
                     dtype=torch.double)
    return WeightedRandomSampler(w, num_samples=int(num_samples), replacement=True)


# ---------------------------------------------------------------------------
# 5. targets + losses
# ---------------------------------------------------------------------------
BOUNDARY_SIGMA = 0.03     # same gaussian boundary targets best_stage2 was trained with


def build_targets_v5(events, vspan, T, device):
    B = len(events)
    centres = (torch.arange(T, dtype=torch.float32) + 0.5) * (W_SEC / T)
    vs = vspan[:, :1]; ve = vspan[:, 1:2]
    valid = ((centres[None] >= vs) & (centres[None] < ve)).float()
    frame = torch.zeros(B, T, NC); on = torch.zeros(B, T); off = torch.zeros(B, T)
    bmask = valid.clone()
    for b in range(B):
        lo, hi = float(vspan[b, 0]), float(vspan[b, 1])
        for s, e, c in events[b]:
            frame[b, (centres >= s) & (centres < e), c] = 1.0
            for edge, tgt in ((s, on), (e, off)):
                if edge <= lo + 1e-3 or edge >= hi - 1e-3:
                    # boundary created by the crop/clip edge: no target, and do not
                    # punish the head near it either
                    bmask[b, (centres - edge).abs() < 0.1] = 0.0
                    continue
                tgt[b] = torch.maximum(tgt[b], torch.exp(-0.5 * ((centres - edge) / BOUNDARY_SIGMA) ** 2))
    return frame.to(device), on.to(device), off.to(device), valid.to(device), bmask.to(device)


_ASL = AsymmetricFocalLoss(gamma_neg=2.0, gamma_pos=1.0, clip=0.05)


def v5_losses(out, batch, cfg, device):
    T = out["T"]
    frame_t, on_t, off_t, valid, bmask = build_targets_v5(batch["events"], batch["vspan"], T, device)
    tier = batch["tier"].to(device)
    strong = ((tier == T_GOLD) | (tier == T_SYNTH)).float()
    sm = valid * strong[:, None]
    cl = out["cat_logits"].float()
    parts = {}
    parts["frame"] = _ASL(cl, frame_t, mask=sm)
    bm = bmask * strong[:, None]
    parts["onset"] = masked_bce(out["onset_logits"].float(), on_t, mask=bm)
    parts["offset"] = masked_bce(out["offset_logits"].float(), off_t, mask=bm)
    p = torch.sigmoid(cl)
    pu = (1.0 - torch.prod(1.0 - p, dim=-1)) * valid
    yu = frame_t.max(-1).values * valid
    d = 1.0 - (2.0 * (pu * yu).sum(1) + 1.0) / (pu.sum(1) + yu.sum(1) + 1.0)
    parts["dice"] = (d * strong).sum() / strong.sum().clamp_min(1.0)
    q = mil_pool(cl, out["attn_logits"], valid)
    z = batch["z"].to(device); zm = batch["zmask"].to(device)
    roww = torch.ones_like(strong); roww[tier == T_BRONZE] = cfg["bronze_weak_w"]
    wz = zm * roww[:, None]
    bce = -(z * torch.log(q) + (1 - z) * torch.log(1 - q))
    parts["weak"] = (bce * wz).sum() / wz.sum().clamp_min(1.0)
    loss = (cfg["w_frame"] * parts["frame"] + cfg["w_on"] * parts["onset"] + cfg["w_off"] * parts["offset"]
            + cfg["w_dice"] * parts["dice"] + cfg["w_weak"] * parts["weak"])
    return loss, parts


# ---------------------------------------------------------------------------
# 6. inference: per-category tracks for a whole clip
# ---------------------------------------------------------------------------
def model_time_grid(net, device):
    net.eval()
    with torch.no_grad():
        o = net(torch.zeros(1, WIN_SAMPLES, device=device), torch.zeros(1, CANON_T, 768, device=device),
                torch.tensor([[0.0, W_SEC]], device=device))
    return o["T"], o["T"] / W_SEC


@torch.no_grad()
def tracks_v5(net, wav, beats_full, device, T_win, fps, max_batch=8):
    """Average logits of overlapping grid windows; returns probabilities."""
    net.eval()
    n = wav.numel(); dur = n / SR
    starts = grid_window_starts(n)
    total_frames = int(round(dur * fps)) + T_win + 1
    acc = np.zeros((total_frames, NC + 2), np.float64); cnt = np.zeros(total_frames)
    for b0 in range(0, len(starts), max_batch):
        bs = starts[b0:b0 + max_batch]
        chunks, rows, gsp = [], [], []
        for s0 in bs:
            ch = wav[s0:s0 + WIN_SAMPLES]
            real = ch.numel()
            if real < WIN_SAMPLES:
                ch = F.pad(ch, (0, WIN_SAMPLES - real))
            r, n_real = beats_rows_for_window(beats_full, s0)
            chunks.append(ch); rows.append(torch.from_numpy(np.ascontiguousarray(r, np.float32)))
            gsp.append((0.0, min(real / SR, n_real * BEATS_PATCH_HOP_SEC)))
        out = net(torch.stack(chunks).to(device), torch.stack(rows).to(device),
                  torch.tensor(gsp, dtype=torch.float32, device=device))
        lg = torch.cat([out["cat_logits"].float(), out["onset_logits"].float()[..., None],
                        out["offset_logits"].float()[..., None]], -1).cpu().numpy()
        for j, s0 in enumerate(bs):
            w = s0 / SR
            off = int(round(w * fps))
            assert abs(off / fps - w) <= 0.5 / fps + 1e-9
            end = min(off + T_win, total_frames); L = end - off
            acc[off:end] += lg[j, :L]; cnt[off:end] += 1
    vl = min(int(round(dur * fps)) + 1, total_frames)
    m = acc[:vl] / np.maximum(cnt[:vl, None], 1e-6)
    pr = 1.0 / (1.0 + np.exp(-m))
    return {"cat": pr[:, :NC].astype(np.float32), "onset": pr[:, NC].astype(np.float32),
            "offset": pr[:, NC + 1].astype(np.float32), "duration": float(dur)}


def ensemble_tracks(list_of_tracks, weights=None):
    weights = weights or [1.0 / len(list_of_tracks)] * len(list_of_tracks)
    out = {k: sum(w * t[k] for w, t in zip(weights, list_of_tracks)) for k in ("cat", "onset", "offset")}
    out["duration"] = list_of_tracks[0]["duration"]
    return out


# ---------------------------------------------------------------------------
# 7. decoder: per category -> flatten
# ---------------------------------------------------------------------------
DEC_DEFAULT = dict(thr_short=0.5, thr_long=0.5, hyst=0.1, sw_short=3, sw_long=9,
                   mindur_short=0.06, mindur_long=0.15, gap_short=0.05, gap_long=0.15,
                   snap=0.10, snap_min=0.2, nms_iou=0.95, max_events=99)


def _runs(mask):
    d = np.diff(np.concatenate([[0], mask.astype(np.int8), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def _snap(probs, idx, radius, pmin):
    lo = max(0, idx - radius); hi = min(len(probs), idx + radius + 1)
    if hi <= lo:
        return idx
    j = lo + int(np.argmax(probs[lo:hi]))
    return j if probs[j] >= pmin else idx


def smooth_track(x, k):
    k = int(k)
    if k <= 1 or len(x) < 3:
        return x
    if k % 2 == 0:
        k += 1
    if k > len(x):
        k = len(x) if len(x) % 2 else len(x) - 1
    return medfilt(x, kernel_size=k) if k >= 3 else x


def decode_v5(tr, p, fps, smoothed=None):
    """smoothed: optional {(c, sw): array} precomputed for speed."""
    dur = tr["duration"]
    radius = int(round(p["snap"] * fps))
    cands = []
    for c in range(NC):
        g = CAT_GROUP[c]
        sw = p[f"sw_{g}"]
        pr = smoothed[(c, sw)] if smoothed is not None else smooth_track(tr["cat"][:, c], sw)
        hi = p[f"thr_{g}"]; lo = max(0.01, hi - p["hyst"])
        evs = []
        for s, e in _runs(pr >= lo):
            if pr[s:e].max() < hi:
                continue
            if radius > 0:
                ss = _snap(tr["onset"], s, radius, p["snap_min"])
                es = _snap(tr["offset"], e, radius, p["snap_min"])
                if es <= ss:
                    ss, es = s, e
            else:
                ss, es = s, e
            evs.append([ss, es])
        evs.sort()
        merged = []
        for s, e in evs:
            if merged and (s - merged[-1][1]) / fps <= p[f"gap_{g}"]:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])
        for s, e in merged:
            if (e - s) / fps >= p[f"mindur_{g}"]:
                cands.append((float(pr[s:max(e, s + 1)].mean()), s, e, c))
    cands.sort(key=lambda x: -x[0])
    kept = []
    for sc, s, e, c in cands:
        ok = True
        for _, ks, ke, _ in kept:
            inter = min(e, ke) - max(s, ks)
            if inter > 0 and inter / (max(e, ke) - min(s, ks)) >= p["nms_iou"]:
                ok = False; break
        if ok:
            kept.append((sc, s, e, c))
        if len(kept) >= p["max_events"]:
            break
    out = []
    for _, s, e, _ in kept:
        on, of = min(s / fps, dur), min(e / fps, dur)
        if of > on:
            out.append((round(on, 3), round(of, 3)))
    return sorted(out)


def precompute_smoothed(tracks_by_key, sws_short, sws_long):
    out = {}
    for k, tr in tracks_by_key.items():
        d = {}
        for c in range(NC):
            for sw in (sws_short if CAT_GROUP[c] == "short" else sws_long):
                d[(c, sw)] = smooth_track(tr["cat"][:, c], sw).astype(np.float32)
        out[k] = d
    return out


def score_sets(dec, sets, fps, smoothed=None, keys=None):
    """sets: {"N": (tracks, refs), "S": (tracks, refs)} -> per-set official scores."""
    res = {}
    for name, (tracks, refs) in sets.items():
        ks = keys[name] if keys is not None else list(tracks)
        pred = {k: decode_v5(tracks[k], dec, fps, smoothed[k] if smoothed is not None else None) for k in ks}
        res[name] = official_combined_score({k: refs[k] for k in ks}, pred)
    return res


W_NAT, W_SYN = 0.62, 0.38            # test mix: 3,437 natural / 2,080 synthetic


def joint(res):
    if "S" not in res:
        return res["N"]["combined"]
    return W_NAT * res["N"]["combined"] + W_SYN * res["S"]["combined"]


def write_submission(pred_by_clip, out_dir, zip_name, durations, expected=None):
    """Writes predictions.jsonl + zip after hard format checks."""
    os.makedirs(out_dir, exist_ok=True)
    if expected is not None and len(pred_by_clip) != expected:
        raise RuntimeError(f"{len(pred_by_clip)} clips predicted, expected {expected}")
    for cid, evs in pred_by_clip.items():
        for on, of in evs:
            assert 0.0 <= on < of <= durations[cid] + 1e-3, (cid, on, of, durations[cid])
    jl = os.path.join(out_dir, zip_name.replace(".zip", ".jsonl"))
    with open(jl, "w", encoding="utf-8") as f:
        for cid in sorted(pred_by_clip):
            f.write(json.dumps({"clip_id": cid, "events": [{"onset": round(float(on), 3), "offset": round(float(of), 3)}
                                                          for on, of in pred_by_clip[cid]]}) + "\n")
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jl, "predictions.jsonl")
    n_ev = sum(len(v) for v in pred_by_clip.values())
    n_empty = sum(1 for v in pred_by_clip.values() if not v)
    print(f"  wrote {zp}: {len(pred_by_clip)} clips, {n_ev} events, {n_empty} empty")
    return zp

print("V5 core definitions loaded.")


## Part 3 — data setup and exhaustive cache validation


In [ ]:
# =============================================================================
# V5 DATA SETUP: dataset, split (same as V3), tiers, labels, exhaustive cache
# validation, district halves. Every later cell uses these objects.
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
set_seed(SEED)
full_dataset = load_from_disk(DATASET_PATH)
N_ROWS = len(full_dataset)
print("rows:", N_ROWS)

train_idx_all, val_idx_all = split_train_val(full_dataset, val_fraction=CONFIG["val_fraction"], seed=CONFIG["seed"])

# ---- exhaustive cache validation: every index except the 2 known-bad ones -----
_missing = [i for i in range(N_ROWS) if resolve_cache_path(i, BEATS_CACHE_DIRS) is None]
_unexpected = sorted(set(_missing) - BAD_INDICES)
print(f"[cache] {N_ROWS - len(_missing)}/{N_ROWS} entries present; missing: {_missing[:10]}")
if _unexpected:
    raise RuntimeError(f"BEATs cache is missing {len(_unexpected)} unexpected entries, first: {_unexpected[:10]}. "
                       f"Fix the cache before running V5 (no live fallback, no substitution).")
train_idx = [i for i in train_idx_all if i not in BAD_INDICES and i not in _missing]
val_idx = [i for i in val_idx_all if i not in BAD_INDICES and i not in _missing]
print(f"split: train {len(train_idx)} | val {len(val_idx)} (excluded {sorted(set(_missing))})")

label_table = build_label_table(full_dataset)
DISTRICT = full_dataset["district"]
DURATION_COL = full_dataset["duration"] if "duration" in full_dataset.column_names else None
build_tag_to_cat(label_table, train_idx)
tiers = split_indices_by_tier(full_dataset, indices=train_idx)
q = label_table["annotationQuality"]
VAL_GOLD = [i for i in val_idx if q[i] == "verified_timestamps"]
print("train tiers:", {k: len(v) for k, v in tiers.items()}, "| val Gold:", len(VAL_GOLD))

# per-event category coverage (should be ~100%: every event carries 'category')
_n_raw = _n_ok = 0
for i in tiers["gold"] + VAL_GOLD:
    raw = label_table["NoiseSubCategoryTimeStamp"][i] or []
    _n_raw += len(raw); _n_ok += len(parse_events(raw, label_table["NoiseCategory"][i]))
print(f"Gold events with a resolved category: {_n_ok}/{_n_raw}")
assert _n_ok >= 0.98 * _n_raw, "category field missing on many Gold events -- check the dataset"

# label space of the old auxiliary heads (needed to load best_stage2.pt unchanged)
canonical_tags, raw_to_canonical, f2c, _ = build_label_space(
    full_dataset, indices=train_idx, min_tag_count_to_keep=CONFIG["min_tag_count_to_keep"])
median_duration = estimate_global_duration_prior(full_dataset, indices=train_idx)
print(f"aux fine tags: {len(canonical_tags)} | median Gold event duration {median_duration:.3f}s")

# district halves of val (same rule as the earlier decoder search: alternate sorted districts)
_val_districts = sorted({DISTRICT[i] for i in VAL_GOLD})
HALF_OF_DISTRICT = {d: ("A" if j % 2 == 0 else "B") for j, d in enumerate(_val_districts)}
print(f"val districts: {len(_val_districts)} -> halves "
      f"A={sum(1 for i in VAL_GOLD if HALF_OF_DISTRICT[DISTRICT[i]] == 'A')} "
      f"B={sum(1 for i in VAL_GOLD if HALF_OF_DISTRICT[DISTRICT[i]] == 'B')} clips")

# test files + ids (used by the leakage guard and the submission cells)
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".m4a")
TEST_FILES = sorted(p for p in glob.glob(os.path.join(TEST_AUDIO_DIR, "*"))
                    if os.path.splitext(p)[1].lower() in AUDIO_EXTS)
TEST_IDS = {os.path.splitext(os.path.basename(p))[0] for p in TEST_FILES}
print(f"test clips found: {len(TEST_FILES)} (expected {EXPECTED_TEST_CLIPS})")
_train_ids = {str(label_table["id"][i]) for i in range(N_ROWS)}
_overlap = TEST_IDS & _train_ids
assert not _overlap, f"{len(_overlap)} test clip ids also appear in the training dataset: {list(_overlap)[:5]}"

V5_DIR = f"{BASE_DIR}/v5_work" + ("_smoke" if SMOKE else "")
_SYN_ROOT = f"{BASE_DIR}/synth_v5" + ("_smoke" if SMOKE else "")
SYNTH_TRAIN_DIR = f"{_SYN_ROOT}/train"
SYNTH_VAL_DIR = f"{_SYN_ROOT}/val"
TEST_BEATS_DIR = f"{BASE_DIR}/beats_cache_test"
SUB_DIR = f"{BASE_DIR}/submission"
for d in (V5_DIR, SUB_DIR):
    os.makedirs(d, exist_ok=True)
LOG_PATH = f"{V5_DIR}/v5_log.txt"
PROGRESS_LIVE_PATH = f"{V5_DIR}/progress_live.txt"


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

log(f"setup done: train {len(train_idx)} val {len(val_idx)} valGold {len(VAL_GOLD)} test {len(TEST_FILES)}")


## Val-N (all Gold val clips) in RAM


In [ ]:
# =============================================================================
# Val-N in RAM: all Gold val clips (waveform + cached BEATs + reference events).
# Used by E0, by every training evaluation and by decoder selection.
# (~1,367 clips, ~1 GB; loading takes a few minutes the first time.)
# =============================================================================
_valn_keys = VAL_GOLD[:120] if SMOKE else VAL_GOLD
VALN = {}
t0 = time.time()
for gi in _valn_keys:
    w = _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS)
    bf = load_nat_beats(gi)
    check_beats_len(bf, w.numel(), f"val:{gi}")
    VALN[str(gi)] = {"wav": w, "beats": bf.astype(np.float16),
                     "ref": [(s, e) for s, e, _ in parse_events(label_table["NoiseSubCategoryTimeStamp"][gi],
                                                               label_table["NoiseCategory"][gi])],
                     "ref_all": _ref_events_for_full_idx(gi, label_table),
                     "half": HALF_OF_DISTRICT[DISTRICT[gi]], "district": DISTRICT[gi]}
# the scorer must see exactly the events the organisers would score: use every valid
# event, even the (rare) ones whose category could not be resolved
VALN_REFS = {k: v["ref_all"] for k, v in VALN.items()}
VALN_HALF = {h: [k for k, v in VALN.items() if v["half"] == h] for h in ("A", "B")}
log(f"Val-N loaded: {len(VALN)} clips in {time.time() - t0:.0f}s "
    f"(A={len(VALN_HALF['A'])}, B={len(VALN_HALF['B'])}); all BEATs lengths match audio")


## E0 — reconcile the inference paths on best_stage2 (+ control / probe zips)


In [ ]:
# =============================================================================
# E0 -- reconcile the three inference paths on best_stage2.pt (no training).
#
#   A        live BEATs per 8 s window (the 0.9965 submission, cell 29)
#   B-floor  cached BEATs, floor() slicing (one patch early on windows >= 2)
#   B-fixed  cached BEATs, windows on the 0.16 s grid, corrected slicing (V5 path)
#
# Every path is decoded with the cell-29 decoder and its exact parameters, on all
# Gold val clips and on each district half. Also defines the control-model helpers
# and the explicit test-BEATs cache builder used by later cells.
# =============================================================================

# ---- cell-29 decoder, verbatim --------------------------------------------
DECODE_C29 = dict(base_thr=0.5, mean_target=0.30, alpha=0.6, thr_min=0.15, thr_max=0.85,
                  clip_gate=0.20, max_p_gate=0.25, smooth_window=3,
                  min_duration=max(0.05, 0.15 * median_duration), merge_gap=0.05,
                  snap_radius_seconds=0.10)


def decode_adaptive(tracks, fps, cfg):
    frame = tracks["frame"]
    max_p = float(frame.max())
    clip_prob = tracks.get("clip_prob", 1.0)
    if clip_prob < cfg["clip_gate"] and max_p < cfg["max_p_gate"]:
        return []
    if max_p < 0.15:
        return []
    mean_p = float(frame.mean())
    thr = cfg["base_thr"] + cfg["alpha"] * (mean_p - cfg["mean_target"])
    thr = max(cfg["thr_min"], min(cfg["thr_max"], thr))
    return decode_events_single(frame, fps, threshold=thr, smooth_window=cfg["smooth_window"],
                                min_duration=cfg["min_duration"], merge_gap=cfg["merge_gap"],
                                onset_probs=tracks["onset"], offset_probs=tracks["offset"],
                                snap_radius_seconds=cfg["snap_radius_seconds"])


_beats_live = None


def get_live_beats():
    global _beats_live
    if _beats_live is None:
        _beats_live, _ = load_beats_model(BEATS_CHECKPOINT, device)
        _beats_live.eval()
    return _beats_live


def load_v3_control():
    ck = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    assert ck["canonical_tags"] == canonical_tags, "label space drifted vs best_stage2.pt"
    mel = MelFrontend(n_mels=CONFIG["n_mels"], sample_rate=SR, max_seconds=W_SEC).to(device)
    m = VaaniLiteSED(get_live_beats(), n_fine=len(canonical_tags), n_coarse=len(COARSE_CATEGORIES),
                     beats_dim=768, mel_frontend=mel).to(device)
    m.load_state_dict(ck["model"])
    m.eval()
    return m, ck


def _merge_windows(outs, starts, T_win, fps, dur):
    total_frames = int(round(dur * fps)) + T_win + 1
    acc = np.zeros((total_frames, 3)); cnt = np.zeros(total_frames); clip = []
    for (fr, on, of, cl), s0 in zip(outs, starts):
        off = int(round((s0 / SR) * fps)); end = min(off + T_win, total_frames); L = end - off
        if L > 0:
            acc[off:end, 0] += fr[:L]; acc[off:end, 1] += on[:L]; acc[off:end, 2] += of[:L]; cnt[off:end] += 1
        clip.append(cl)
    vl = min(int(round(dur * fps)) + 1, total_frames)
    m = acc[:vl] / np.maximum(cnt[:vl, None], 1e-6)
    pr = 1.0 / (1.0 + np.exp(-m))
    return {"frame": pr[:, 0], "onset": pr[:, 1], "offset": pr[:, 2],
            "clip_prob": float(1.0 / (1.0 + np.exp(-np.mean(clip)))), "duration": dur}


def _v3_out(out):
    return (out["event_frame_logits"][0, :, 0].float().cpu().numpy(),
            out["event_onset_logits"][0, :, 0].float().cpu().numpy(),
            out["event_offset_logits"][0, :, 0].float().cpu().numpy(),
            float(out["event_clip_logits"][0, 0]))


@torch.no_grad()
def v3_tracks(model, wav, beats_full, path, T_win, fps):
    n = wav.numel(); dur = n / SR; outs = []
    if path == "B-fixed":
        starts = grid_window_starts(n)
    else:
        starts = _window_starts(n, WIN_SAMPLES, STRIDE_SAMPLES)
    for s0 in starts:
        ch = wav[s0:s0 + WIN_SAMPLES]; real = ch.numel()
        if real < WIN_SAMPLES:
            ch = F.pad(ch, (0, WIN_SAMPLES - real))
        ch = ch.unsqueeze(0).to(device)
        if path == "A":
            vr = torch.tensor([real / WIN_SAMPLES], device=device)
            out = model(ch, beats_cached=None, beats_valid_ratio=vr)
        elif path == "B-floor":       # V4 tracks_core slicing
            if n <= WIN_SAMPLES:
                bw = beats_full
            else:
                T_full = beats_full.shape[0]
                p0 = int(np.floor((s0 / SR - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC))
                p1 = int(np.ceil(((s0 + WIN_SAMPLES) / SR - BEATS_PATCH_OFFSET_SEC) / BEATS_PATCH_HOP_SEC)) + 1
                p0 = max(0, min(p0, T_full - 1)); p1 = max(p0 + 1, min(p1, T_full))
                bw = beats_full[p0:p1]
            if bw.shape[0] < CANON_T:
                bw = np.concatenate([bw, np.zeros((CANON_T - bw.shape[0], bw.shape[1]), bw.dtype)], 0)
            bw = torch.from_numpy(np.ascontiguousarray(bw[:CANON_T], np.float32)).unsqueeze(0).to(device)
            out = model(ch, beats_cached=bw, beats_valid_ratio=torch.tensor([real / WIN_SAMPLES], device=device))
        else:                          # B-fixed
            rows, n_real = beats_rows_for_window(beats_full, s0)
            bw = torch.from_numpy(np.ascontiguousarray(rows, np.float32)).unsqueeze(0).to(device)
            vr = min(real / SR, n_real * BEATS_PATCH_HOP_SEC) / W_SEC
            out = model(ch, beats_cached=bw, beats_valid_ratio=torch.tensor([vr], device=device))
        outs.append(_v3_out(out))
    return _merge_windows(outs, starts, T_win, fps, dur)


def score_c29(tracks_by_key, refs, fps, keys=None):
    ks = keys if keys is not None else list(tracks_by_key)
    pred = {k: decode_adaptive(tracks_by_key[k], fps, DECODE_C29) for k in ks}
    return official_combined_score({k: refs[k] for k in ks}, pred)


def build_test_beats_cache():
    """EXPLICIT live extraction for the test clips only (same fp32 extractor and
    fp16 storage as the training cache). Resumable; hard-fails on bad output."""
    os.makedirs(TEST_BEATS_DIR, exist_ok=True)
    bm = get_live_beats(); n_new = 0
    for j, pth in enumerate(TEST_FILES):
        cid = os.path.splitext(os.path.basename(pth))[0]
        out = os.path.join(TEST_BEATS_DIR, f"{cid}.npy")
        if os.path.exists(out):
            continue
        w = load_test_wav(pth)
        if w.numel() < SR:     # BEATs needs >= ~0.2 s; pad very short clips to 1 s for extraction only
            print(f"  [test-beats] {cid}: {w.numel() / SR:.2f}s clip, padded to 1 s for BEATs")
            w = F.pad(w, (0, SR - w.numel()))
        feat, _ = extract_beats_full(bm, w, device)
        if not np.isfinite(feat).all():
            raise RuntimeError(f"non-finite BEATs output for test clip {cid}")
        check_beats_len(feat, w.numel(), f"test:{cid}")
        np.save(out, feat.astype(np.float16)); n_new += 1
        if (j + 1) % 500 == 0:
            print(f"  [test-beats] {j + 1}/{len(TEST_FILES)}")
    log(f"test BEATs cache: {n_new} new, {len(TEST_FILES)} total in {TEST_BEATS_DIR}")


def load_test_wav(pth):
    x, sr0 = sf.read(pth, dtype="float32")
    if x.ndim > 1:
        x = x.mean(axis=1)
    w = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))
    if sr0 != SR:
        w = torchaudio.functional.resample(w, int(sr0), SR)
    return w.contiguous()


def load_test_beats(cid, n_samples):
    p = os.path.join(TEST_BEATS_DIR, f"{cid}.npy")
    if not os.path.exists(p):
        raise RuntimeError(f"missing test BEATs entry {p}: run build_test_beats_cache() first")
    bf = np.load(p).astype(np.float32)
    check_beats_len(bf, max(n_samples, SR), f"test:{cid}")
    return bf


E0_PATH = f"{V5_DIR}/e0_tracks.pkl"
if RUN_E0_RECONCILE:
    v3, _ck = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    log(f"E0: best_stage2 stored combined={_ck['combined']:.4f}; time grid T={T3} fps={fps3:.3f}")
    E0 = {}
    for path in ("A", "B-floor", "B-fixed"):
        t0 = time.time()
        E0[path] = {k: v3_tracks(v3, v["wav"], v["beats"].astype(np.float32), path, T3, fps3)
                    for k, v in VALN.items()}
        r_all = score_c29(E0[path], VALN_REFS, fps3)
        r_a = score_c29(E0[path], VALN_REFS, fps3, VALN_HALF["A"])
        r_b = score_c29(E0[path], VALN_REFS, fps3, VALN_HALF["B"])
        log(f"E0 {path:8s}: all={r_all['combined']:.4f} (F1 {r_all['event_f1']:.4f} Dice {r_all['segment_dice']:.4f}) "
            f"A={r_a['combined']:.4f} B={r_b['combined']:.4f} [{time.time() - t0:.0f}s]")
    with open(E0_PATH, "wb") as f:
        pickle.dump({"fps": fps3, "tracks": E0}, f)
    a = score_c29(E0["A"], VALN_REFS, fps3)["combined"]
    bf_ = score_c29(E0["B-fixed"], VALN_REFS, fps3)["combined"]
    bl = score_c29(E0["B-floor"], VALN_REFS, fps3)["combined"]
    log(f"E0 verdict: B-fixed - A = {bf_ - a:+.4f}; B-fixed - B-floor = {bf_ - bl:+.4f}. "
        + ("Cached+fixed inference is at least as good as the live path -> safe for test."
           if bf_ >= a - 0.005 else
           "Cached+fixed is worse than live on val: keep the live path for the control; "
           "V5 is trained on the fixed cache so it uses B-fixed regardless."))
    del v3; torch.cuda.empty_cache() if torch.cuda.is_available() else None

if RUN_CONTROL_ZIP or RUN_E0_TEST_PROBE:
    v3, _ = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    if RUN_E0_TEST_PROBE:
        build_test_beats_cache()
    for path, flag, name in (("A", RUN_CONTROL_ZIP, "submission_0p9965_control.zip"),
                             ("B-fixed", RUN_E0_TEST_PROBE, "submission_E0_bfixed.zip")):
        if not flag:
            continue
        preds, durs = {}, {}
        for pth in TEST_FILES:
            cid = os.path.splitext(os.path.basename(pth))[0]
            w = load_test_wav(pth)
            bf = load_test_beats(cid, w.numel()) if path == "B-fixed" else None
            tr = v3_tracks(v3, w, bf, path, T3, fps3)
            preds[cid] = decode_adaptive(tr, fps3, DECODE_C29); durs[cid] = tr["duration"]
        write_submission(preds, SUB_DIR, name, durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    del v3


## E1 — synthetic mixtures and Val-S


In [ ]:
# =============================================================================
# E1 -- synthetic mixtures ("clean speech + added noise") with exact labels.
#
# Sources are ONLY training-dataset clips (plus ESC-50 if you enable it):
#   bed  70%: clean speech = event-free stretches (>=1 s, 0.1 s margin) of Gold clips
#        30%: a whole Gold clip, keeping its own labelled events
#   fg   Gold event cuts (category from the annotation), category-balanced (p ~ n^0.5)
#   K = 1/2/3 events (p .55/.30/.15), SNR U[-5,+15] dB vs the bed's speech RMS,
#   5 ms fades, peak-limited to 0.99. Labels: onset = paste position, offset =
#   paste position + pasted length (exact by construction).
# Train mixtures come from TRAIN-district Gold clips only; Val-S comes from
# VAL-district Gold clips only, so no audio is shared between the two.
# Output: <root>/wav/<id>.flac, <root>/beats/<id>.npy (fp16), manifest.jsonl.
# Resumable: items already in the manifest are skipped.
# =============================================================================
N_SYNTH_TRAIN = 300 if SMOKE else 24000
N_SYNTH_VAL = 80 if SMOKE else 1000
SNR_RANGE = (-5.0, 15.0)
K_PROBS = (0.55, 0.30, 0.15)
FG_DUR_RANGE = (0.08, 6.0)
CLEAN_BED_PROB = 0.7
ESC_MAP = {  # ESC-50 class -> Vaani category (used only if ESC50_DIR is set)
    "dog": "animal", "rooster": "animal", "pig": "animal", "cow": "animal", "frog": "animal",
    "cat": "animal", "hen": "animal", "insects": "animal", "sheep": "animal", "crow": "animal",
    "chirping_birds": "animal",
    "crying_baby": "baby_child",
    "sneezing": "human_non_speech", "clapping": "human_non_speech", "breathing": "human_non_speech",
    "coughing": "human_non_speech", "laughing": "human_non_speech", "snoring": "human_non_speech",
    "footsteps": "human_non_speech", "drinking_sipping": "human_non_speech", "brushing_teeth": "human_non_speech",
    "car_horn": "vehicle_traffic", "engine": "vehicle_traffic", "train": "vehicle_traffic",
    "airplane": "vehicle_traffic", "helicopter": "vehicle_traffic",
    "vacuum_cleaner": "appliance_machine", "washing_machine": "appliance_machine",
    "hand_saw": "appliance_machine", "chainsaw": "appliance_machine", "keyboard_typing": "appliance_machine",
    "mouse_click": "appliance_machine",
    "clock_alarm": "phone_signal_alarm", "siren": "phone_signal_alarm", "church_bells": "phone_signal_alarm",
}


class _WavLRU:
    def __init__(self, cap=256):
        self.cap = cap; self.d = {}
    def get(self, gi):
        if gi in self.d:
            return self.d[gi]
        w = _load_waveform(gi, full_dataset, SR, WAV_CACHE_DIRS).numpy()
        if len(self.d) >= self.cap:
            self.d.pop(next(iter(self.d)))
        self.d[gi] = w
        return w


def _active_rms(x):
    if len(x) < 400:
        return float(np.sqrt(np.mean(x ** 2)) + 1e-8)
    fr = x[: len(x) // 160 * 160].reshape(-1, 160)
    e = np.sqrt((fr ** 2).mean(1) + 1e-12)
    act = e[e > e.max() * 10 ** (-40 / 20)]
    return float(np.sqrt(np.mean(act ** 2)) + 1e-8)


def _fade(x, n=80):
    n = min(n, len(x) // 2)
    if n > 0:
        r = np.linspace(0, 1, n, dtype=np.float32)
        x = x.copy(); x[:n] *= r; x[-n:] *= r[::-1]
    return x


def _esc_bank():
    if not ESC50_DIR:
        return []
    import csv as _csv
    meta = os.path.join(ESC50_DIR, "meta", "esc50.csv")
    bank = []
    with open(meta) as f:
        for r in _csv.DictReader(f):
            if r["category"] in ESC_MAP:
                bank.append({"path": os.path.join(ESC50_DIR, "audio", r["filename"]),
                             "cat": ESC_MAP[r["category"]], "fold": int(r["fold"])})
    print(f"  ESC-50 foregrounds available: {len(bank)}")
    return bank


def _esc_cut(path):
    x, sr0 = sf.read(path, dtype="float32")
    if x.ndim > 1:
        x = x.mean(1)
    if sr0 != SR:
        x = torchaudio.functional.resample(torch.from_numpy(x), sr0, SR).numpy()
    fr = x[: len(x) // 160 * 160].reshape(-1, 160)
    e = 20 * np.log10(np.sqrt((fr ** 2).mean(1)) + 1e-9)
    act = np.where(e > e.max() - 35)[0]
    if len(act) == 0:
        return None
    return x[act[0] * 160:(act[-1] + 1) * 160]


def build_synth(root, prefix, n_items, gold_indices, seed, esc_folds):
    os.makedirs(os.path.join(root, "wav"), exist_ok=True)
    os.makedirs(os.path.join(root, "beats"), exist_ok=True)
    store = SynthStore(root)
    done = set(store.items)
    # assertion: no test clip and no clip from the other split can be a source
    for gi in gold_indices:
        assert str(label_table["id"][gi]) not in TEST_IDS
    lru = _WavLRU()
    # clean-speech beds and foreground bank
    beds, fg, gold_clips = [], defaultdict(list), []
    for gi in gold_indices:
        raw = label_table["NoiseSubCategoryTimeStamp"][gi] or []
        evs = parse_events(raw, label_table["NoiseCategory"][gi])
        d = float(DURATION_COL[gi]) if DURATION_COL is not None else len(lru.get(gi)) / SR
        if len(evs) == len([1 for e in raw if e]):           # every event resolved
            t = 0.0
            for s, e, _ in sorted(evs):
                if s - 0.1 - t >= 1.0:
                    beds.append((gi, t, s - 0.1))
                t = max(t, e + 0.1)
            if d - t >= 1.0:
                beds.append((gi, t, d))
            if 2.0 <= d <= 14.0:
                gold_clips.append((gi, d, evs))
        for s, e, c in evs:
            if FG_DUR_RANGE[0] <= e - s <= FG_DUR_RANGE[1]:
                fg[c].append(("gold", gi, s, e))
    esc = [b for b in _esc_bank() if b["fold"] in esc_folds]
    for b in esc:
        fg[CAT_IDX[b["cat"]]].append(("esc", b["path"], 0.0, 0.0))
    cats_avail = [c for c in range(NC) if fg[c]]
    pc = np.array([len(fg[c]) ** 0.5 for c in cats_avail]); pc /= pc.sum()
    durs = np.array([d for _, d, _ in gold_clips]) if gold_clips else np.array([6.0])
    print(f"  [{prefix}] beds={len(beds)} fg per cat={ {CATS[c]: len(fg[c]) for c in range(NC)} } "
          f"whole-clip beds={len(gold_clips)} | already built {len(done)}/{n_items}")
    bm = None
    t0 = time.time(); n_new = 0
    mf = open(os.path.join(root, "manifest.jsonl"), "a", encoding="utf-8")
    try:
        for n in range(n_items):
            sid = f"{prefix}{n:06d}"
            if sid in done:
                continue
            rng = np.random.default_rng([seed, n])
            # ---- bed ----
            if rng.random() < CLEAN_BED_PROB or not gold_clips:
                D = float(np.clip(rng.choice(durs), 2.0, 14.0))
                parts, src, L = [], [], 0
                while L < D * SR:
                    gi, a, b = beds[int(rng.integers(len(beds)))]
                    w = lru.get(gi)
                    seg = _fade(w[int(a * SR):int(b * SR)])
                    parts.append(seg); src.append([int(gi), round(a, 3), round(b, 3)]); L += len(seg)
                bed = np.concatenate(parts)[: int(D * SR)].astype(np.float32)
                bed_events, mode = [], "clean"
            else:
                gi, D, evs = gold_clips[int(rng.integers(len(gold_clips)))]
                bed = lru.get(gi).astype(np.float32).copy()
                D = len(bed) / SR
                bed_events = [(s, min(e, D), CATS[c]) for s, e, c in evs if min(e, D) > s]
                src, mode = [[int(gi), 0.0, round(D, 3)]], "gold"
            D = len(bed) / SR
            speech_rms = _active_rms(bed)
            # ---- foregrounds ----
            K = int(rng.choice([1, 2, 3], p=K_PROBS))
            pasted, mix = [], bed.copy()
            for _ in range(K):
                c = cats_avail[int(rng.choice(len(cats_avail), p=pc))]
                kind, a, s, e = fg[c][int(rng.integers(len(fg[c])))]
                if kind == "gold":
                    seg = lru.get(a)[int(s * SR):int(e * SR)]
                else:
                    seg = _esc_cut(a)
                    if seg is None:
                        continue
                if len(seg) < int(0.05 * SR):
                    continue
                max_len = int(0.8 * D * SR)
                if len(seg) > max_len:
                    o = int(rng.integers(0, len(seg) - max_len + 1)); seg = seg[o:o + max_len]
                pos = int(rng.integers(0, len(mix) - len(seg) + 1))
                snr = float(rng.uniform(*SNR_RANGE))
                gain = speech_rms / (_active_rms(seg) * 10 ** (snr / 20))
                mix[pos:pos + len(seg)] += _fade(seg.astype(np.float32)) * gain
                pasted.append({"cat": CATS[c], "src": [kind, a if kind == "esc" else int(a), round(s, 4), round(e, 4)],
                               "pos": pos, "len": int(len(seg)), "snr": round(snr, 2), "gain": round(float(gain), 5)})
            if not pasted:
                continue
            pk = float(np.abs(mix).max())
            if pk > 0.99:
                mix *= 0.99 / pk
            events = [(round(s, 4), round(e, 4), c) for s, e, c in bed_events]
            events += [(round(p["pos"] / SR, 4), round((p["pos"] + p["len"]) / SR, 4), p["cat"]) for p in pasted]
            sf.write(os.path.join(root, "wav", f"{sid}.flac"), mix, SR, format="FLAC", subtype="PCM_16")
            w_back, _ = sf.read(os.path.join(root, "wav", f"{sid}.flac"), dtype="float32")
            if bm is None:
                bm = get_live_beats()
            feat, _ = extract_beats_full(bm, torch.from_numpy(w_back), device)
            if not np.isfinite(feat).all():
                raise RuntimeError(f"non-finite BEATs output for {sid}")
            np.save(os.path.join(root, "beats", f"{sid}.npy"), feat.astype(np.float16))
            rec = {"id": sid, "duration": round(len(w_back) / SR, 4), "mode": mode, "bed_src": src,
                   "district": DISTRICT[src[0][0]], "events": events, "pasted": pasted}
            mf.write(json.dumps(rec) + "\n"); mf.flush()
            n_new += 1
            if n_new % 1000 == 0:
                print(f"    [{prefix}] {n_new} new ({time.time() - t0:.0f}s)")
    finally:
        mf.close()
    log(f"synthetic {prefix}: {n_new} new mixtures in {time.time() - t0:.0f}s -> {root}")


def verify_synth(root, gold_set):
    """Labels re-derived from the stored recipe must equal the stored labels; sources
    must belong to the intended split; files must exist with the stated length."""
    st = SynthStore(root)
    for sid, r in st.items.items():
        exp = [(round(p["pos"] / SR, 4), round((p["pos"] + p["len"]) / SR, 4), p["cat"]) for p in r["pasted"]]
        got = [tuple(e) for e in r["events"]][-len(exp):]
        assert [tuple(x) for x in exp] == [(a, b, c) for a, b, c in got], f"label mismatch {sid}"
        for g in r["bed_src"]:
            assert g[0] in gold_set, f"{sid} uses a bed outside its split"
        for p in r["pasted"]:
            if p["src"][0] == "gold":
                assert p["src"][1] in gold_set, f"{sid} uses a foreground outside its split"
        for s, e, c in r["events"]:
            assert 0 <= s < e <= r["duration"] + 1e-3 and c in CAT_IDX, (sid, s, e, c)
    print(f"  verified {len(st.items)} items in {root}")
    return st


if RUN_BUILD_SYNTH:
    _tr_gold = [i for i in tiers["gold"]]
    build_synth(SYNTH_TRAIN_DIR, "s", N_SYNTH_TRAIN, _tr_gold, seed=SEED, esc_folds={1, 2, 3, 4})
    build_synth(SYNTH_VAL_DIR, "v", N_SYNTH_VAL, VAL_GOLD, seed=SEED + 1, esc_folds={5})
SYNTH_TRAIN = verify_synth(SYNTH_TRAIN_DIR, set(tiers["gold"])) if os.path.exists(SYNTH_TRAIN_DIR) else None
SYNTH_VAL = verify_synth(SYNTH_VAL_DIR, set(VAL_GOLD)) if os.path.exists(SYNTH_VAL_DIR) else None


## E2 — train V5-A


In [ ]:
# =============================================================================
# E2 -- train V5-A (warm start from best_stage2.pt).
#
# Model selection uses ALL Gold val clips (Val-N) and all Val-S mixtures:
#   J = 0.62 * Combined(Val-N) + 0.38 * Combined(Val-S)
# with a light threshold sweep (the submission decoder is selected later, E3).
# Real optimiser steps are counted only when AMP did not skip the step; the LR
# schedule and EMA advance only on real steps.
# =============================================================================
VALS = {}
if SYNTH_VAL is not None:
    for sid in sorted(SYNTH_VAL.items)[: (60 if SMOKE else None)]:
        r = SYNTH_VAL.items[sid]
        w = SYNTH_VAL.wav(sid); bf = SYNTH_VAL.beats(sid)
        check_beats_len(bf, w.numel(), f"valS:{sid}")
        VALS[sid] = {"wav": w, "beats": bf.astype(np.float16), "ref": [(s, e) for s, e, _ in r["events"]],
                     "half": HALF_OF_DISTRICT.get(r["district"], "A")}
VALS_REFS = {k: v["ref"] for k, v in VALS.items()}
VALS_HALF = {h: [k for k, v in VALS.items() if v["half"] == h] for h in ("A", "B")}
log(f"Val-S loaded: {len(VALS)} mixtures (A={len(VALS_HALF['A'])}, B={len(VALS_HALF['B'])})")
if not VALS:
    print("WARNING: no Val-S -- selection will use Val-N only")

CFG_V5 = dict(lr=1e-4, min_lr=1e-6, weight_decay=1e-2, warmup_updates=300, total_updates=8000,
              eval_every=1000, batch_size=CONFIG["batch_size"], grad_accum_steps=CONFIG["grad_accum_steps"],
              ema_decay=0.999, early_stop_evals=4,
              w_frame=3.0, w_on=0.5, w_off=0.5, w_dice=0.5, w_weak=0.5, bronze_weak_w=0.5,
              shares={T_GOLD: 0.35, T_SYNTH: 0.30, T_SILVER: 0.25, T_BRONZE: 0.10},
              shift_prob=0.8, beats_mask_prob=0.5, feat_dropout=0.2)
if SMOKE:
    CFG_V5.update(total_updates=30, eval_every=15, warmup_updates=5)

mel_v5 = MelFrontend(n_mels=CONFIG["n_mels"], sample_rate=SR, max_seconds=W_SEC).to(device)


def new_v5():
    return VaaniSEDV5(n_fine=len(canonical_tags), mel_frontend=mel_v5,
                      feat_dropout=CFG_V5["feat_dropout"]).to(device)


def v5_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = new_v5(); net.load_state_dict(ck["model"]); net.eval()
    return net, ck


def eval_tracks(net, valset, T_win, fps):
    return {k: tracks_v5(net, v["wav"], v["beats"].astype(np.float32), device, T_win, fps)
            for k, v in valset.items()}


SEL_DEC = dict(DEC_DEFAULT)


def selection_J(tracks_n, tracks_s, fps):
    """Light, threshold-robust selection score (same threshold for every category)."""
    best = None
    for thr in (0.3, 0.4, 0.5, 0.6, 0.7):
        dec = dict(SEL_DEC, thr_short=thr, thr_long=thr)
        sets = {"N": (tracks_n, VALN_REFS)}
        if tracks_s:
            sets["S"] = (tracks_s, VALS_REFS)
        res = score_sets(dec, sets, fps)
        J = joint(res)
        if best is None or J > best["J"]:
            best = {"J": J, "thr": thr, "N": res["N"]["combined"], "S": res["S"]["combined"] if "S" in res else None,
                    "N_f1": res["N"]["event_f1"], "N_dice": res["N"]["segment_dice"]}
    return best


def train_v5(rows, run_tag, total_updates, do_eval=True, init_state=None):
    set_seed(SEED)
    model = new_v5()
    if init_state is not None:
        model.init_from_v3(init_state)
    ema = new_v5(); ema.load_state_dict(model.state_dict()); freeze_model(ema)
    T_win, fps = model_time_grid(model, device)
    assert T_win == 801, f"unexpected time grid {T_win}"
    ds = V5Dataset(rows, full_dataset, label_table, SYNTH_TRAIN, training=True,
                   shift_prob=CFG_V5["shift_prob"], beats_mask_prob=CFG_V5["beats_mask_prob"])
    n_draw = (total_updates + 50) * CFG_V5["batch_size"] * CFG_V5["grad_accum_steps"]
    loader = DataLoader(ds, batch_size=CFG_V5["batch_size"], sampler=make_tier_sampler(rows, CFG_V5["shares"], n_draw),
                        num_workers=0, collate_fn=collate_v5, drop_last=True, pin_memory=(device.type == "cuda"))
    params = model.trainable_parameters()
    opt = torch.optim.AdamW(params, lr=CFG_V5["lr"], weight_decay=CFG_V5["weight_decay"])
    floor = CFG_V5["min_lr"] / CFG_V5["lr"]

    def lr_at(u):
        wu = CFG_V5["warmup_updates"]
        if u < wu:
            return 0.1 + 0.9 * u / wu
        pr = min(1.0, (u - wu) / max(1, total_updates - wu))
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * pr))

    for g in opt.param_groups:
        g["lr"] = CFG_V5["lr"] * lr_at(0)
    scaler = GradScaler(enabled=(device.type == "cuda"))
    accum = CFG_V5["grad_accum_steps"]
    paths = {"student": f"{V5_DIR}/{run_tag}_student.pt", "ema": f"{V5_DIR}/{run_tag}_ema.pt"}
    best = {"student": -1.0, "ema": -1.0}; best_info = {}
    hist = []
    if do_eval:
        b0 = selection_J(eval_tracks(model, VALN, T_win, fps), eval_tracks(model, VALS, T_win, fps), fps)
        log(f"{run_tag} start (warm start from best_stage2 via 7 copied heads): J={b0['J']:.4f} "
            f"N={b0['N']:.4f} S={b0['S'] if b0['S'] is None else round(b0['S'], 4)} thr={b0['thr']}")
        hist.append({"update": 0, "ema": b0})
    upd = micro = skipped = 0; run = defaultdict(float); evals_since = 0; t0 = time.time()
    model.train()
    opt.zero_grad(set_to_none=True)
    for batch in loader:
        wav = batch["waveform"].to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=False):
            mel = model.mel_frontend(wav.float())
        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            out = model(wav, batch["beats"].to(device, non_blocking=True), batch["gspan"].to(device),
                        mel=weak_augment(mel))
        loss, parts = v5_losses(out, batch, CFG_V5, device)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite loss {({k: float(v) for k, v in parts.items()})} keys={batch['key']}")
        scaler.scale(loss / accum).backward()
        for k, v in parts.items():
            run[k] += float(v.detach())
        run["n"] += 1; micro += 1
        if micro % accum:
            continue
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        prev = scaler.get_scale()
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if scaler.get_scale() < prev:          # AMP skipped this step (inf/nan grads)
            skipped += 1
            continue
        upd += 1
        for g in opt.param_groups:
            g["lr"] = CFG_V5["lr"] * lr_at(upd)
        update_teacher(model, ema, decay=CFG_V5["ema_decay"])
        if upd % 100 == 0:
            with open(PROGRESS_LIVE_PATH, "w") as f:
                f.write(f"run={run_tag}\nupdate={upd}/{total_updates}\nskipped={skipped}\n"
                        f"lr={opt.param_groups[0]['lr']:.2e}\nloss={float(loss):.4f}\n"
                        f"elapsed_min={(time.time() - t0) / 60:.1f}\n")
        last = upd >= total_updates
        if (upd % CFG_V5["eval_every"] == 0) or last:
            tr = " ".join(f"{k}={v / run['n']:.4f}" for k, v in run.items() if k != "n"); run = defaultdict(float)
            line = f"{run_tag} upd={upd} skipped={skipped} lr={opt.param_groups[0]['lr']:.2e} {tr}"
            if do_eval:
                improved = False
                for name, net in (("student", model), ("ema", ema)):
                    sc = selection_J(eval_tracks(net, VALN, T_win, fps), eval_tracks(net, VALS, T_win, fps), fps)
                    line += f" | {name}: J={sc['J']:.4f} N={sc['N']:.4f} S={sc['S'] if sc['S'] is None else round(sc['S'], 4)}"
                    if sc["J"] > best[name]:
                        best[name] = sc["J"]; improved = True; best_info[name] = {"update": upd, **sc}
                        sd = net.state_dict()
                        torch.save({"model": sd, "update": upd, "score": sc, "hash": state_hash(sd),
                                    "canonical_tags": canonical_tags, "cfg": CFG_V5}, paths[name])
                        line += " *saved*"
                    hist.append({"update": upd, name: sc})
                evals_since = 0 if improved else evals_since + 1
            model.train()
            log(line)
            if do_eval and evals_since >= CFG_V5["early_stop_evals"]:
                log(f"{run_tag}: early stop, no improvement in {evals_since} evaluations"); break
        if upd >= total_updates:
            break
    if upd < total_updates and evals_since < CFG_V5["early_stop_evals"]:
        raise RuntimeError(f"loader exhausted early at {upd}/{total_updates} updates")
    if not do_eval:
        for name, net in (("student", model), ("ema", ema)):
            sd = net.state_dict()
            torch.save({"model": sd, "update": upd, "hash": state_hash(sd), "canonical_tags": canonical_tags,
                        "cfg": CFG_V5}, paths[name])
    with open(f"{V5_DIR}/{run_tag}_history.json", "w") as f:
        json.dump({"best": best_info, "hist": hist, "updates": upd, "skipped": skipped}, f, default=float)
    log(f"{run_tag} done: {upd} updates ({skipped} AMP-skipped) in {(time.time() - t0) / 60:.1f} min; best {best_info}")
    return paths, best_info


TRAIN_ROWS = ([("nat", i, T_GOLD) for i in tiers["gold"]] + [("nat", i, T_SILVER) for i in tiers["silver"]]
              + [("nat", i, T_BRONZE) for i in tiers["bronze"]]
              + ([("syn", s, T_SYNTH) for s in sorted(SYNTH_TRAIN.items)] if SYNTH_TRAIN is not None else []))
_val_set = set(VAL_GOLD)
assert not any(r[0] == "nat" and r[1] in _val_set for r in TRAIN_ROWS), "val clip leaked into training rows"

if RUN_TRAIN_V5:
    _v3 = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    assert _v3["canonical_tags"] == canonical_tags
    V5_PATHS, V5_BEST = train_v5(TRAIN_ROWS, RUN_TAG, CFG_V5["total_updates"], do_eval=True, init_state=_v3["model"])
    del _v3


## E3 — decoder selection and control comparison


In [ ]:
# =============================================================================
# E3 -- decoder selection for the chosen V5 checkpoint, and the control comparison.
#
# Rule: objective J = 0.62*Comb(Val-N) + 0.38*Comb(Val-S).
#   1. random search (N_CFG configs) scored on district half A only
#   2. the top 15 on A are re-scored on half B
#   3. accepted only if it beats DEC_DEFAULT on BOTH halves; else DEC_DEFAULT
# The control (best_stage2 + cell-29 decoder, live path A) is scored on the same
# halves. V5 should only replace the control on the LB if it beats it on both.
# =============================================================================
N_CFG = 40 if SMOKE else 300
SW_SHORT = [1, 3, 5, 9]
SW_LONG = [3, 5, 9, 15, 25]


def pick_v5_checkpoint(run_tag):
    cands = []
    for name in ("ema", "student"):
        p = f"{V5_DIR}/{run_tag}_{name}.pt"
        if os.path.exists(p):
            ck = torch.load(p, map_location="cpu", weights_only=False)
            cands.append((ck.get("score", {}).get("J", -1), name, p))
    if not cands:
        raise RuntimeError(f"no V5 checkpoint for {run_tag} in {V5_DIR} -- run the training cell")
    cands.sort(reverse=True)
    print("  checkpoints:", [(n, round(j, 4)) for j, n, _ in cands])
    return cands[0][2]


def sample_dec(r):
    return dict(thr_short=float(r.uniform(0.2, 0.8)), thr_long=float(r.uniform(0.2, 0.8)),
                hyst=float(r.choice([0.0, 0.05, 0.1, 0.2])),
                sw_short=int(r.choice(SW_SHORT)), sw_long=int(r.choice(SW_LONG)),
                mindur_short=float(r.uniform(0.02, 0.15)), mindur_long=float(r.uniform(0.05, 0.4)),
                gap_short=float(r.choice([0.0, 0.03, 0.05, 0.1])), gap_long=float(r.choice([0.0, 0.05, 0.1, 0.2, 0.3])),
                snap=float(r.choice([0.0, 0.05, 0.1, 0.15, 0.25])), snap_min=float(r.choice([0.1, 0.2, 0.3])),
                nms_iou=float(r.choice([0.5, 0.7, 0.95, 1.01])), max_events=int(r.choice([2, 3, 5, 99])))


def half_scores(dec, TN, TS, fps, SMN, SMS):
    out = {}
    for h in ("A", "B"):
        sets = {"N": (TN, VALN_REFS)}; keys = {"N": VALN_HALF[h]}; sm = dict(SMN)
        if TS:
            sets["S"] = (TS, VALS_REFS); keys["S"] = VALS_HALF[h]; sm.update(SMS)
        out[h] = joint(score_sets(dec, sets, fps, sm, keys))
    return out


def select_decoder(TN, TS, fps, tag):
    SMN = precompute_smoothed(TN, SW_SHORT, SW_LONG)
    SMS = precompute_smoothed(TS, SW_SHORT, SW_LONG) if TS else {}
    base = half_scores(DEC_DEFAULT, TN, TS, fps, SMN, SMS)
    log(f"{tag} DEC_DEFAULT: A={base['A']:.4f} B={base['B']:.4f}")
    r = np.random.default_rng(SEED)
    scored = []
    t0 = time.time()
    for i in range(N_CFG):
        d = sample_dec(r)
        sets = {"N": (TN, VALN_REFS)}; keys = {"N": VALN_HALF["A"]}; sm = dict(SMN)
        if TS:
            sets["S"] = (TS, VALS_REFS); keys["S"] = VALS_HALF["A"]; sm.update(SMS)
        scored.append((joint(score_sets(d, sets, fps, sm, keys)), i, d))
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{N_CFG} configs ({time.time() - t0:.0f}s), best A so far {max(scored, key=lambda x: x[0])[0]:.4f}")
    scored.sort(key=lambda x: -x[0])
    chosen, chosen_s, searched = dict(DEC_DEFAULT), base, False
    for ja, _, d in scored[:15]:
        hs = half_scores(d, TN, TS, fps, SMN, SMS)
        if hs["A"] > base["A"] and hs["B"] > base["B"] and hs["B"] > chosen_s["B"]:
            chosen, chosen_s, searched = d, hs, True
    log(f"{tag} selected decoder: A={chosen_s['A']:.4f} B={chosen_s['B']:.4f} "
        f"({'searched config' if searched else 'DEC_DEFAULT kept: no searched config beat it on both halves'}) {chosen}")
    return chosen, chosen_s


def control_half_scores(fps3, trA_N, trA_S):
    out = {}
    for h in ("A", "B"):
        rn = score_c29(trA_N, VALN_REFS, fps3, VALN_HALF[h])["combined"]
        rs = score_c29(trA_S, VALS_REFS, fps3, VALS_HALF[h])["combined"] if trA_S else None
        out[h] = rn if rs is None else W_NAT * rn + W_SYN * rs
    return out


if RUN_DECODER_SELECTION:
    ck_path = pick_v5_checkpoint(RUN_TAG)
    net, ck = v5_from_ckpt(ck_path)
    T_win, fps = model_time_grid(net, device)
    TN = eval_tracks(net, VALN, T_win, fps); TS = eval_tracks(net, VALS, T_win, fps)
    DEC_V5, DEC_V5_S = select_decoder(TN, TS, fps, RUN_TAG)
    full = joint(score_sets(DEC_V5, {"N": (TN, VALN_REFS), **({"S": (TS, VALS_REFS)} if TS else {})}, fps))
    with open(f"{V5_DIR}/{RUN_TAG}_decoder.json", "w") as f:
        json.dump({"decoder": DEC_V5, "halves": DEC_V5_S, "J_full": full, "checkpoint": ck_path,
                   "hash": ck["hash"]}, f, indent=1)
    with open(f"{V5_DIR}/{RUN_TAG}_val_tracks.pkl", "wb") as f:
        pickle.dump({"fps": fps, "N": TN, "S": TS, "hash": ck["hash"]}, f)
    del net

    # ---- control: best_stage2 + cell-29 decoder, live path A ----------------
    v3, _ = load_v3_control()
    T3, fps3 = get_model_time_grid(v3, device, W_SEC, SR)
    if os.path.exists(E0_PATH):
        with open(E0_PATH, "rb") as f:
            trA_N = pickle.load(f)["tracks"]["A"]
        trA_N = {k: trA_N[k] for k in VALN}
    else:
        trA_N = {k: v3_tracks(v3, v["wav"], None, "A", T3, fps3) for k, v in VALN.items()}
    trA_S = {k: v3_tracks(v3, v["wav"], None, "A", T3, fps3) for k, v in VALS.items()}
    del v3
    CTRL_S = control_half_scores(fps3, trA_N, trA_S)
    v5_wins = DEC_V5_S["A"] > CTRL_S["A"] and DEC_V5_S["B"] > CTRL_S["B"]
    log(f"CONTROL (0.9965 path) on val: A={CTRL_S['A']:.4f} B={CTRL_S['B']:.4f} | "
        f"V5 {RUN_TAG}: A={DEC_V5_S['A']:.4f} B={DEC_V5_S['B']:.4f} -> "
        + ("V5 beats the control on both halves: submit V5." if v5_wins else
           "V5 does NOT beat the control on both halves: keep the control as your main submission."))


## Test submission (V5)


In [ ]:
# =============================================================================
# Test submission for V5 (path B-fixed: cached test BEATs + corrected slicing).
# 1. builds the test BEATs cache EXPLICITLY (resumable, hard-fails on bad output)
# 2. runs the selected checkpoint, caches test tracks (re-decode later on CPU)
# 3. decodes with the decoder selected for THAT checkpoint (hash-checked)
# 4. writes submission/submission_<RUN_TAG>.zip after format checks
# =============================================================================
def run_test_submission(run_tag, ck_path, dec_json_path, zip_name):
    with open(dec_json_path) as f:
        dj = json.load(f)
    net, ck = v5_from_ckpt(ck_path)
    if dj["hash"] != ck["hash"]:
        print(f"  NOTE: decoder was selected on checkpoint {dj['hash']}, applying it to {ck['hash']} "
              f"({'expected for the final retrain' if 'final' in run_tag else 'UNEXPECTED'})")
        assert "final" in run_tag, "decoder/checkpoint mismatch"
    T_win, fps = model_time_grid(net, device)
    build_test_beats_cache()
    preds, durs, tracks = {}, {}, {}
    t0 = time.time()
    for j, pth in enumerate(TEST_FILES):
        cid = os.path.splitext(os.path.basename(pth))[0]
        w = load_test_wav(pth)
        bf = load_test_beats(cid, w.numel())
        tr = tracks_v5(net, w, bf, device, T_win, fps)
        tracks[cid] = {k: (v.astype(np.float16) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}
        preds[cid] = decode_v5(tr, dj["decoder"], fps); durs[cid] = tr["duration"]
        if (j + 1) % 500 == 0:
            print(f"  [test] {j + 1}/{len(TEST_FILES)} ({time.time() - t0:.0f}s)")
    with open(f"{V5_DIR}/{run_tag}_test_tracks.pkl", "wb") as f:
        pickle.dump({"fps": fps, "tracks": tracks, "hash": ck["hash"]}, f)
    zp = write_submission(preds, SUB_DIR, zip_name, durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
    n_ev = np.array([len(v) for v in preds.values()])
    log(f"{run_tag} test submission {zp}: events/clip mean {n_ev.mean():.2f}, empty {int((n_ev == 0).sum())}")
    return zp


if RUN_TEST_SUBMISSION:
    _dj = f"{V5_DIR}/{RUN_TAG}_decoder.json"
    if not os.path.exists(_dj):
        raise RuntimeError("run the decoder-selection cell first (it writes the decoder for this checkpoint)")
    with open(_dj) as f:
        _ck_path = json.load(f)["checkpoint"]
    run_test_submission(RUN_TAG, _ck_path, _dj, f"submission_{RUN_TAG}{'_SMOKE' if SMOKE else ''}.zip")


## E7 — optional final retrain on all Gold


In [ ]:
# =============================================================================
# E7 (optional) -- final retrain on ALL Gold (train + val) + Silver/Bronze + synthetic,
# warm-started from best_stage2.pt, for the SAME number of updates at which the E2
# run's selected checkpoint peaked. No selection happens here (there is no held-out
# data left), so the decoder chosen in E3 is reused. Submit this as a separate LB
# entry and keep whichever of the two scores better.
# =============================================================================
if RUN_FINAL_RETRAIN:
    with open(f"{V5_DIR}/{RUN_TAG}_decoder.json") as f:
        _dj = json.load(f)
    _sel = torch.load(_dj["checkpoint"], map_location="cpu", weights_only=False)
    _n_upd = int(_sel["update"])
    assert _n_upd > 0, "selected checkpoint is the untrained start -- nothing to retrain"
    _which = "ema" if _dj["checkpoint"].endswith("_ema.pt") else "student"
    log(f"E7: retraining for {_n_upd} updates (selected {_which} checkpoint peaked there)")
    _rows = TRAIN_ROWS + [("nat", i, T_GOLD) for i in VAL_GOLD]
    _v3 = torch.load(f"{BASE_DIR}/best_stage2.pt", map_location="cpu", weights_only=False)
    final_tag = f"{RUN_TAG}_final"
    _paths, _ = train_v5(_rows, final_tag, _n_upd, do_eval=False, init_state=_v3["model"])
    _final_dec = f"{V5_DIR}/{final_tag}_decoder.json"
    with open(_final_dec, "w") as f:
        json.dump(dict(_dj, checkpoint=_paths[_which]), f, indent=1)
    run_test_submission(final_tag, _paths[_which], _final_dec,
                        f"submission_{final_tag}{'_SMOKE' if SMOKE else ''}.zip")


In [ ]:
# =============================================================================
# DECODER-2 -- CPU only, no training. Re-decodes the CACHED tracks of an existing
# run (default v5a) with three new decoder options aimed at what the diagnosis
# found (boundary misses and merged events):
#   split_{g}  split a run at a valley deeper than ratio * run peak  (merges)
#   edge_{g}   place each edge where the track crosses rel * run peak (boundaries)
#   shon_{g} / shoff_{g}  constant onset / offset shift            (bias: offsets late)
# Same rule as E3: search on district half A (0.62 Val-N + 0.38 Val-S), confirm on
# half B, accept only if it beats the run's current decoder on BOTH halves.
# If accepted, it re-decodes the cached TEST tracks (no GPU, no BEATs) and writes
# submission/submission_<tag>_dec2.zip -- one leaderboard probe, one change.
# Run this single cell after setup + Val-N + training cell (all RUN_* flags off).
# =============================================================================
# ---- paste-in version for an EXISTING kernel: redefines decode_v5 with the new
# ---- options (old decoder configs decode exactly as before), then runs Decoder-2.
def _split_valleys(pr, s, e, ratio, hi, min_len):
    """Split run [s,e) at local minima deeper than ratio * run peak, keeping only pieces
    that still reach hi. ratio <= 0 disables."""
    if ratio <= 0 or e - s < 2 * min_len + 1:
        return [(s, e)]
    peak = float(pr[s:e].max()); cut = ratio * peak
    pieces, cur = [], s
    i = s + min_len
    while i < e - min_len:
        if pr[i] < cut and pr[i] <= pr[i - 1] and pr[i] <= pr[i + 1] \
                and pr[cur:i].max() >= hi and pr[i + 1:e].max() >= hi:
            pieces.append((cur, i)); cur = i + 1; i = cur + min_len
            continue
        i += 1
    pieces.append((cur, e))
    return pieces


def _rel_edges(pr, s, e, rel, max_ext):
    """Move edges to where pr crosses rel * (run peak): shrink inside the run, or extend
    outward by at most max_ext frames. rel <= 0 disables."""
    if rel <= 0:
        return s, e
    t = rel * float(pr[s:e].max()); n = len(pr)
    if pr[s] >= t:
        lim = max(0, s - max_ext)
        while s > lim and pr[s - 1] >= t:
            s -= 1
    else:
        while s < e - 1 and pr[s] < t:
            s += 1
    if pr[e - 1] >= t:
        lim = min(n, e + max_ext)
        while e < lim and pr[e] >= t:
            e += 1
    else:
        while e > s + 1 and pr[e - 1] < t:
            e -= 1
    return s, e


def decode_v5(tr, p, fps, smoothed=None):
    """smoothed: optional {(c, sw): array} precomputed for speed.
    Decoder-2 keys (all optional; missing = off, so older configs decode identically):
      split_{g}: valley-split ratio, edge_{g}: relative-edge level,
      shon_{g} / shoff_{g}: constant onset / offset shift in seconds."""
    dur = tr["duration"]
    radius = int(round(p["snap"] * fps))
    max_ext = int(round(0.3 * fps)); min_piece = max(2, int(round(0.05 * fps)))
    n_fr = tr["cat"].shape[0]
    cands = []
    for c in range(NC):
        g = CAT_GROUP[c]
        sw = p[f"sw_{g}"]
        pr = smoothed[(c, sw)] if smoothed is not None else smooth_track(tr["cat"][:, c], sw)
        hi = p[f"thr_{g}"]; lo = max(0.01, hi - p["hyst"])
        split_r = p.get(f"split_{g}", 0.0); edge_r = p.get(f"edge_{g}", 0.0)
        sh_on = int(round(p.get(f"shon_{g}", 0.0) * fps)); sh_off = int(round(p.get(f"shoff_{g}", 0.0) * fps))
        evs = []
        for s0, e0 in _runs(pr >= lo):
            if pr[s0:e0].max() < hi:
                continue
            for s, e in _split_valleys(pr, s0, e0, split_r, hi, min_piece):
                s, e = _rel_edges(pr, s, e, edge_r, max_ext)
                if radius > 0:
                    on_tr = tr["onset"][:, c] if tr["onset"].ndim == 2 else tr["onset"]
                    of_tr = tr["offset"][:, c] if tr["offset"].ndim == 2 else tr["offset"]
                    ss = _snap(on_tr, s, radius, p["snap_min"])
                    es = _snap(of_tr, e, radius, p["snap_min"])
                    if es <= ss:
                        ss, es = s, e
                else:
                    ss, es = s, e
                ss = min(max(0, ss + sh_on), n_fr); es = min(max(0, es + sh_off), n_fr)
                if es > ss:
                    evs.append([ss, es])
        evs.sort()
        merged = []
        for s, e in evs:
            if merged and (s - merged[-1][1]) / fps <= p[f"gap_{g}"]:
                merged[-1][1] = max(merged[-1][1], e)
            else:
                merged.append([s, e])
        for s, e in merged:
            if (e - s) / fps >= p[f"mindur_{g}"]:
                cands.append((float(pr[s:max(e, s + 1)].mean()), s, e, c))
    cands.sort(key=lambda x: -x[0])
    kept = []
    for sc, s, e, c in cands:
        ok = True
        for _, ks, ke, _ in kept:
            inter = min(e, ke) - max(s, ks)
            if inter > 0 and inter / (max(e, ke) - min(s, ks)) >= p["nms_iou"]:
                ok = False; break
        if ok:
            kept.append((sc, s, e, c))
        if len(kept) >= p["max_events"]:
            break
    out = []
    for _, s, e, _ in kept:
        on, of = min(s / fps, dur), min(e / fps, dur)
        if of > on:
            out.append((round(on, 3), round(of, 3)))
    return sorted(out)


RUN_DECODER2 = True
DEC2_TAG = "v5a"
N_CFG2 = 40 if SMOKE else 400

if RUN_DECODER2:
    with open(f"{V5_DIR}/{DEC2_TAG}_val_tracks.pkl", "rb") as f:
        vt = pickle.load(f)
    with open(f"{V5_DIR}/{DEC2_TAG}_decoder.json") as f:
        dj = json.load(f)
    assert vt["hash"] == dj["hash"], "val tracks and decoder come from different checkpoints"
    fps2, TN2, TS2 = vt["fps"], vt["N"], vt["S"]
    TN2 = {k: v for k, v in TN2.items() if k in VALN_REFS}
    TS2 = {k: v for k, v in TS2.items() if k in VALS_REFS}
    SMN2 = precompute_smoothed(TN2, SW_SHORT, SW_LONG); SMS2 = precompute_smoothed(TS2, SW_SHORT, SW_LONG)
    old = dict(dj["decoder"])
    base2 = half_scores(old, TN2, TS2, fps2, SMN2, SMS2)
    log(f"DECODER-2 {DEC2_TAG}: current decoder A={base2['A']:.4f} B={base2['B']:.4f}")

    def sample_dec2(r):
        d = sample_dec(r) if r.random() < 0.5 else dict(old)       # half fresh, half around the current one
        if d is old or d == old:
            d = dict(old)
            for k in ("thr_short", "thr_long"):
                d[k] = float(np.clip(d[k] + r.normal(0, 0.05), 0.1, 0.9))
        for g in ("short", "long"):
            d[f"split_{g}"] = float(r.choice([0.0, 0.0, 0.5, 0.6, 0.7, 0.8]))
            d[f"edge_{g}"] = float(r.choice([0.0, 0.0, 0.3, 0.4, 0.5, 0.6]))
            d[f"shon_{g}"] = float(r.choice([0.0, 0.0, -0.04, -0.02, 0.02, 0.04]))
            d[f"shoff_{g}"] = float(r.choice([0.0, 0.0, -0.08, -0.05, -0.03, 0.03]))
        return d

    r2 = np.random.default_rng(SEED + 7)
    scored2 = []; t0 = time.time()
    for i in range(N_CFG2):
        d = sample_dec2(r2)
        res = score_sets(d, {"N": (TN2, VALN_REFS), "S": (TS2, VALS_REFS)}, fps2,
                         {**SMN2, **SMS2}, {"N": VALN_HALF["A"], "S": VALS_HALF["A"]})
        scored2.append((joint(res), i, d))
        if (i + 1) % 100 == 0:
            print(f"    {i + 1}/{N_CFG2} ({time.time() - t0:.0f}s) best A {max(x[0] for x in scored2):.4f}")
    scored2.sort(key=lambda x: -x[0])
    best2, best2_s = None, base2
    for ja, _, d in scored2[:20]:
        hs = half_scores(d, TN2, TS2, fps2, SMN2, SMS2)
        if hs["A"] > base2["A"] and hs["B"] > base2["B"] and hs["B"] > best2_s["B"]:
            best2, best2_s = d, hs
    if best2 is None:
        log("DECODER-2: no config beat the current decoder on both halves -> nothing to submit")
    else:
        log(f"DECODER-2 selected: A={best2_s['A']:.4f} (+{best2_s['A'] - base2['A']:.4f}) "
            f"B={best2_s['B']:.4f} (+{best2_s['B'] - base2['B']:.4f}) {best2}")
        refsN2 = {k: parse_events(label_table["NoiseSubCategoryTimeStamp"][int(k)], label_table["NoiseCategory"][int(k)])
                  for k in TN2}
        refsS2 = {sid: [(s, e, CAT_IDX[c]) for s, e, c in SYNTH_VAL.items[sid]["events"]] for sid in TS2}
        for name, TR, RC in (("Val-N", TN2, refsN2), ("Val-S", TS2, refsS2)):
            _report(f"{name} dec2", RC, {k: decode_v5(TR[k], best2, fps2) for k in RC})
        with open(f"{V5_DIR}/{DEC2_TAG}_decoder2.json", "w") as f:
            json.dump(dict(dj, decoder=best2, halves=best2_s, previous=old), f, indent=1)
        tp = f"{V5_DIR}/{DEC2_TAG}_test_tracks.pkl"
        with open(tp, "rb") as f:
            tt = pickle.load(f)
        assert tt["hash"] == dj["hash"], "cached test tracks come from a different checkpoint"
        preds2, durs2 = {}, {}
        for cid, tr in tt["tracks"].items():
            tr = {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}
            preds2[cid] = decode_v5(tr, best2, tt["fps"]); durs2[cid] = tr["duration"]
        zp = write_submission(preds2, SUB_DIR, f"submission_{DEC2_TAG}_dec2{'_SMOKE' if SMOKE else ''}.zip",
                              durs2, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
        ne = np.array([len(v) for v in preds2.values()])
        log(f"DECODER-2 test zip {zp}: events/clip {ne.mean():.2f}, empty {int((ne == 0).sum())}")


In [ ]:
# =============================================================================
# COUNT PROBES -- CPU only. Re-decodes the CACHED v5a test tracks with the v5a
# decoder (the 1.2047 submission) changed in exactly ONE way per zip, to measure on
# the leaderboard whether the test set wants FEWER predicted events than our val.
# (Both submissions that predicted more events per test clip -- BND 2.36, dec2 2.30 --
# lost to v5a at 2.06, while local val said the opposite.)
# Submit them one at a time; each zip name says what changed.
# =============================================================================
RUN_PROBES = True
PROBE_TAG = "v5a"
PROBES = {
    "max2":   {"max_events": 2},
    "thr+05": {"thr_short": +0.05, "thr_long": +0.05},      # relative change
    "max1":   {"max_events": 1},
}

if RUN_PROBES:
    with open(f"{V5_DIR}/{PROBE_TAG}_decoder.json") as f:
        _dj = json.load(f)
    with open(f"{V5_DIR}/{PROBE_TAG}_test_tracks.pkl", "rb") as f:
        _tt = pickle.load(f)
    assert _tt["hash"] == _dj["hash"], "cached test tracks come from a different checkpoint"
    _tracks = {cid: {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}
               for cid, tr in _tt["tracks"].items()}
    _base = dict(_dj["decoder"])
    _ref_pred = {cid: decode_v5(tr, _base, _tt["fps"]) for cid, tr in _tracks.items()}
    with open(f"{V5_DIR}/{PROBE_TAG}_val_tracks.pkl", "rb") as f:
        _vt = pickle.load(f)
    _sets = {"N": (_vt["N"], VALN_REFS), **({"S": (_vt["S"], VALS_REFS)} if VALS_REFS else {})}
    _keys = {"N": [k for k in _vt["N"] if k in VALN_REFS], **({"S": [k for k in _vt["S"] if k in VALS_REFS]} if VALS_REFS else {})}
    log(f"PROBE base ({PROBE_TAG} decoder): test events/clip {np.mean([len(v) for v in _ref_pred.values()]):.2f} | "
        f"local J {joint(score_sets(_base, _sets, _vt['fps'], keys=_keys)):.4f}")
    for name, ch in PROBES.items():
        d = dict(_base)
        for k, v in ch.items():
            d[k] = (d[k] + v) if k.startswith("thr_") else v
        preds = {cid: decode_v5(tr, d, _tt["fps"]) for cid, tr in _tracks.items()}
        durs = {cid: tr["duration"] for cid, tr in _tracks.items()}
        zp = write_submission(preds, SUB_DIR, f"submission_{PROBE_TAG}_probe_{name}{'_SMOKE' if SMOKE else ''}.zip",
                              durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
        ne = np.array([len(v) for v in preds.values()])
        _res = score_sets(d, _sets, _vt["fps"], keys=_keys)     # same change on local val
        log(f"PROBE {name}: test events/clip {ne.mean():.2f} (empty {int((ne == 0).sum())}) | local J {joint(_res):.4f} "
            f"N {_res['N']['combined']:.4f} S {_res['S']['combined'] if 'S' in _res else float('nan'):.4f} -> {zp}")


In [ ]:
# =============================================================================
# FIXED-DECODER MODEL COMPARISON (v2) -- CPU only, no training, no decoder search.
#
# Judges a candidate model against V5-A with the decoder held FIXED at the V5-A
# decoder that produced 1.2047 on the leaderboard. FAILS CLOSED: any missing
# input, schema mismatch or coverage mismatch raises instead of producing a verdict.
#
# Predeclared accept rule (set before any candidate result was seen):
#   Val-N Combined gain >= MIN_GAIN on district half A AND on half B,
#   Val-N aggregate gain >= MIN_GAIN, and Val-S drop no worse than -MAX_S_DROP.
# Provenance: the fixed decoder must reproduce submission_v5a.zip exactly from the
# cached v5a test tracks, so it is provably the 1.2047 decoder.
# Output if accepted: submission/submission_<CAND>_v5adec.zip
# Needs: the diagnose cell (for _match) to have been run in this kernel.
# =============================================================================
RUN_FIXED_COMPARE = True
BASE_TAG = "v5a"
CAND_TAG = "v5c_smel"
MIN_GAIN = 0.005        # per natural half and aggregate
MAX_S_DROP = 0.02


def _load_pkl(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} missing: run training + decoder selection + test submission for that tag")
    with open(p, "rb") as f:
        return pickle.load(f)


def _check_val_schema(v, tag):
    for k in ("fps", "N", "S", "hash"):
        assert k in v, f"{tag} val pickle lacks '{k}' (keys: {list(v)})"
    tr = next(iter(v["N"].values()))
    assert {"cat", "onset", "offset", "duration"} <= set(tr), f"{tag} val track keys {list(tr)}"
    assert tr["cat"].ndim == 2 and tr["cat"].shape[1] == NC, f"{tag} cat shape {tr['cat'].shape}"


def _check_test_schema(t, tag):
    for k in ("fps", "tracks", "hash"):
        assert k in t, f"{tag} test pickle lacks '{k}' (keys: {list(t)})"
    tr = next(iter(t["tracks"].values()))
    assert {"cat", "onset", "offset", "duration"} <= set(tr), f"{tag} test track keys {list(tr)}"


def _test_tracks32(tag):
    """Test tracks exactly as decoded for the zip. Older runs cached them as float16,
    which shifts onset/offset snapping (argmax near-ties) -> regenerate in float32 once
    from the same checkpoint (GPU, ~3 min) and keep as <tag>_test_tracks32.pkl."""
    p32 = f"{V5_DIR}/{tag}_test_tracks32.pkl"
    if os.path.exists(p32):
        return _load_pkl(p32)
    t = _load_pkl(f"{V5_DIR}/{tag}_test_tracks.pkl")
    _check_test_schema(t, tag)
    if next(iter(t["tracks"].values()))["cat"].dtype == np.float32:
        return t
    with open(f"{V5_DIR}/{tag}_decoder.json") as f:
        ck_path = json.load(f)["checkpoint"]
    net, ck = v5_from_ckpt(ck_path)
    assert ck["hash"] == t["hash"], f"{tag}: checkpoint {ck_path} is not the one that produced the cached test tracks"
    T_win, fps = model_time_grid(net, device)
    log(f"FIXED-DEC regenerating float32 test tracks for {tag} from {os.path.basename(ck_path)} ...")
    out = {}
    for pth in TEST_FILES:
        cid = os.path.splitext(os.path.basename(pth))[0]
        w = load_test_wav(pth)
        out[cid] = tracks_v5(net, w, load_test_beats(cid, w.numel()), device, T_win, fps)
    res = {"fps": fps, "tracks": out, "hash": ck["hash"]}
    with open(p32, "wb") as f:
        pickle.dump(res, f)
    del net
    return res


def _f32(tr):
    return {k: (v.astype(np.float32) if isinstance(v, np.ndarray) else v) for k, v in tr.items()}


def _short_recall(tracks, refs_cat, dec, fps):
    missing = [k for k in refs_cat if k not in tracks]
    if missing:
        raise RuntimeError(f"short-event recall: {len(missing)} reference clips have no track, e.g. {missing[:3]}")
    tot = hit = 0
    for k, rc in refs_cat.items():
        pairs = _match([(s, e) for s, e, _ in rc], decode_v5(tracks[k], dec, fps))
        matched = {r for r, _ in pairs}
        for ri, (s, e, _) in enumerate(rc):
            if e - s < 0.3:
                tot += 1; hit += ri in matched
    return hit / max(tot, 1), tot


if RUN_FIXED_COMPARE:
    assert "_match" in globals(), "run the diagnose cell first (it defines _match)"
    with open(f"{V5_DIR}/{BASE_TAG}_decoder.json") as f:
        _bj = json.load(f)
    DEC_FIXED = dict(_bj["decoder"])
    dec_sha = hashlib.sha1(json.dumps(DEC_FIXED, sort_keys=True).encode()).hexdigest()[:12]
    V = {t: _load_pkl(f"{V5_DIR}/{t}_val_tracks.pkl") for t in (BASE_TAG, CAND_TAG)}
    TT = {t: _test_tracks32(t) for t in (BASE_TAG, CAND_TAG)}
    for t in V:
        _check_val_schema(V[t], t); _check_test_schema(TT[t], t)
    assert V[BASE_TAG]["hash"] == _bj["hash"] == TT[BASE_TAG]["hash"], "v5a val/test tracks and decoder disagree"
    assert abs(V[BASE_TAG]["fps"] - V[CAND_TAG]["fps"]) < 1e-9
    fpsx = V[BASE_TAG]["fps"]

    # ---- provenance: fixed decoder + cached v5a test tracks == submission_v5a.zip
    _bzip = f"{SUB_DIR}/submission_{BASE_TAG}{'_SMOKE' if SMOKE else ''}.zip"
    with zipfile.ZipFile(_bzip) as z:
        _sub = {json.loads(l)["clip_id"]: [(e["onset"], e["offset"]) for e in json.loads(l)["events"]]
                for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()}
    _redo = {cid: decode_v5(_f32(tr), DEC_FIXED, TT[BASE_TAG]["fps"]) for cid, tr in TT[BASE_TAG]["tracks"].items()}
    assert set(_sub) == set(_redo), f"{_bzip} and the cached v5a test tracks cover different clips"
    _same = lambda a, b: len(a) == len(b) and all(abs(x[0] - y[0]) < 1e-6 and abs(x[1] - y[1]) < 1e-6 for x, y in zip(a, b))
    _diff = [c for c in _sub if not _same(_sub[c], _redo[c])]
    if len(_diff) > 0.01 * len(_sub):
        raise RuntimeError(f"fixed decoder does not reproduce {_bzip}: {len(_diff)}/{len(_sub)} clips differ")
    log(f"FIXED-DEC decoder sha {dec_sha} reproduces {os.path.basename(_bzip)}: {len(_sub) - len(_diff)}/{len(_sub)} "
        f"clips identical" + (f" ({len(_diff)} differ: GPU non-determinism in regenerated tracks, <1% tolerated)" if _diff else ""))

    # ---- identical coverage, fail closed
    if not VALS_REFS:
        raise RuntimeError("Val-S references not loaded -- synthetic comparison would be skipped; refusing")
    for t in V:
        for name, refs in (("N", VALN_REFS), ("S", VALS_REFS)):
            miss = [k for k in refs if k not in V[t][name]]
            if miss:
                raise RuntimeError(f"{t}: {len(miss)} {name} validation clips missing from cached tracks, e.g. {miss[:3]}")
    TT_ids = {t: set(TT[t]["tracks"]) for t in TT}
    assert TT_ids[BASE_TAG] == TT_ids[CAND_TAG] and len(TT_ids[CAND_TAG]) == (len(TEST_FILES) if not SMOKE else len(TT_ids[CAND_TAG]))
    refsN = {k: parse_events(label_table["NoiseSubCategoryTimeStamp"][int(k)], label_table["NoiseCategory"][int(k)])
             for k in VALN_REFS}
    keys = {h: {"N": list(VALN_HALF[h]), "S": list(VALS_HALF[h])} for h in ("A", "B")}
    keys["all"] = {"N": list(VALN_REFS), "S": list(VALS_REFS)}
    for h in keys:
        assert keys[h]["N"] and keys[h]["S"], f"empty subset for {h}"
    log(f"FIXED-DEC coverage: Val-N {len(VALN_REFS)} (A {len(keys['A']['N'])}, B {len(keys['B']['N'])}), "
        f"Val-S {len(VALS_REFS)} (A {len(keys['A']['S'])}, B {len(keys['B']['S'])}), test {len(TT_ids[CAND_TAG])} -- identical for both")

    rows = {}
    for t in (BASE_TAG, CAND_TAG):
        r = {h: score_sets(DEC_FIXED, {"N": (V[t]["N"], VALN_REFS), "S": (V[t]["S"], VALS_REFS)}, fpsx, keys=keys[h])
             for h in keys}
        r["epc"] = np.mean([len(decode_v5(V[t]["N"][k], DEC_FIXED, fpsx)) for k in VALN_REFS])
        r["short"], n_short = _short_recall(V[t]["N"], refsN, DEC_FIXED, fpsx)
        rows[t] = r
        a = r["all"]
        log(f"FIXED-DEC [{t}] Val-N {a['N']['combined']:.4f} (F1 {a['N']['event_f1']:.4f} P {a['N']['precision']:.3f} "
            f"R {a['N']['recall']:.3f} Dice {a['N']['segment_dice']:.4f}) A {r['A']['N']['combined']:.4f} "
            f"B {r['B']['N']['combined']:.4f} | Val-S {a['S']['combined']:.4f} (F1 {a['S']['event_f1']:.4f} "
            f"Dice {a['S']['segment_dice']:.4f}) | pred/clip {r['epc']:.2f} | short<0.3s recall {r['short']:.3f} (n={n_short})")
    b, c = rows[BASE_TAG], rows[CAND_TAG]
    d = {h: c[h]["N"]["combined"] - b[h]["N"]["combined"] for h in ("A", "B", "all")}
    dS = c["all"]["S"]["combined"] - b["all"]["S"]["combined"]
    ok = d["A"] >= MIN_GAIN and d["B"] >= MIN_GAIN and d["all"] >= MIN_GAIN and dS >= -MAX_S_DROP
    log(f"FIXED-DEC verdict {CAND_TAG} vs {BASE_TAG} (decoder {dec_sha}, MIN_GAIN {MIN_GAIN}): dValN A {d['A']:+.4f} "
        f"B {d['B']:+.4f} all {d['all']:+.4f}, dValS {dS:+.4f} -> "
        + ("ACCEPT: submit the zip below." if ok else "REJECT: keep v5a; do not submit."))
    if ok:
        preds, durs = {}, {}
        for cid, tr in TT[CAND_TAG]["tracks"].items():
            tr = _f32(tr)
            preds[cid] = decode_v5(tr, DEC_FIXED, TT[CAND_TAG]["fps"]); durs[cid] = tr["duration"]
        zp = write_submission(preds, SUB_DIR, f"submission_{CAND_TAG}_v5adec{'_SMOKE' if SMOKE else ''}.zip",
                              durs, expected=None if SMOKE else EXPECTED_TEST_CLIPS)
        ne = np.array([len(v) for v in preds.values()])
        log(f"FIXED-DEC zip {zp} (decoder {dec_sha}): test events/clip {ne.mean():.2f}, empty {int((ne == 0).sum())}")
